# Original residual-add + FP32 RMSNorm — MICROBENCH_ONLY

Native expanded Qwen add/norm vs the immutable original historical standalone norm vs new original two-output residual/norm fusion. Not another standalone full-task trial. No model, training, optimizer, score, ownership-share or admission. No model hook or installed solution. The FP32 residual and FP32 norm output remain separate; no FP16 narrowing, in-place alias mutation or extra gradients.

Ordinary engineering gates retain prior trial001 tolerances: forward rtol1e-4/atol1e-5; gradient after known synthetic-scale normalization rtol1e-3/atol2e-5; finite/nonfinite classes exact. These are not official task-quality gates. Both-output, norm-only, residual-only, same/overlapping input, one-required-input and saved-version cases are checked. Known extreme expanded-autograd branch association differs (native0 vs atomic1 on CPU); diagnostic is explicitly retained, so universal equality is NOT claimed. Partial preflight arrays remain exported on failure.

Source inspection identifies the natural Qwen post-attention residual-add then norm boundary. No production replacement is implemented; it would require an exact guarded decoder-forward adaptation and careful hooks/wrapper review. Cross-layer add fusion is not proposed.

Each GPU child retains80s, protected runtime/source/lock/guardian/quarantine/idle helpers. Private empty per-method compiler caches; no prewarming across processes. Five observed T values133/235/317/485/512 with synthetic B4,H1536 FP32 tensors. Fixed native/standalone/fused order. First-call wall includes compile/allocation/dispatch/device work; warm CUDA-event stream intervals include host/autograd gaps, not pure kernel sums. No full-training speed or GPU-memory claim. No downloads/installs/credentials/automatic retry. Parent alone submits after independent review.


In [ ]:
import os,base64,hashlib,tempfile
from pathlib import Path
assert os.environ.get('SOUP_REPO') and os.environ.get('SOUP_RESULTS')
assert Path('/kaggle/working').is_dir()
PAYLOAD = {'solution/__init__.py': 'IiIiT3JpZ2luYWwgcmVzaWR1YWwtYWRkL1JNU05vcm0gY29tcG9uZW50IHJlc2VhcmNoLCBubyBtb2RlbCBpbnN0YWxsYXRpb24uIiIiCg==', 'solution/contracts.py': 'IiIiT3JpZ2luYWwgc2hhcmVkIG51bWVyaWNhbCBmaXh0dXJlczsgbm8gdGFyZ2V0IGltcG9ydCB1bnRpbCBjYWxsZWQuIiIiCmltcG9ydCB0b3JjaApIPTE1MzYKCmRlZiBuYXRpdmUoYSxiLHcsZXBzPTFlLTYpOgogICAgej1hK2IKICAgIHZhcmlhbmNlPXoucG93KDIpLm1lYW4oLTEsa2VlcGRpbT1UcnVlKQogICAgcmV0dXJuIHcqKHoqdG9yY2gucnNxcnQodmFyaWFuY2UrZXBzKSksegoKZGVmIHN0YWdlZF9keChkeSx6LHcsaW52ZXJzZSk6CiAgICBnPWR5Knc7ZGlyZWN0PWcqaW52ZXJzZTtkaW52ZXJzZT0oZyp6KS5zdW0oLTEsa2VlcGRpbT1UcnVlKQogICAgZHZhcmlhbmNlPSgtMC41KmRpbnZlcnNlKSooKGludmVyc2UqaW52ZXJzZSkqaW52ZXJzZSkKICAgIHJldHVybiBkaXJlY3QrKGR2YXJpYW5jZS96LnNoYXBlWy0xXSkqKDIqeikKCmNsYXNzIENwdUJvdW5kYXJ5KHRvcmNoLmF1dG9ncmFkLkZ1bmN0aW9uKToKICAgIEBzdGF0aWNtZXRob2QKICAgIGRlZiBmb3J3YXJkKGN0eCxhLGIsdyk6CiAgICAgICAgej1hK2I7aW52ZXJzZT10b3JjaC5yc3FydCh6LnNxdWFyZSgpLm1lYW4oLTEsa2VlcGRpbT1UcnVlKSsxZS02KQogICAgICAgIGN0eC5zYXZlX2Zvcl9iYWNrd2FyZCh6LHcsaW52ZXJzZSk7Y3R4LnNldF9tYXRlcmlhbGl6ZV9ncmFkcyhGYWxzZSkKICAgICAgICByZXR1cm4gdyooeippbnZlcnNlKSx6CiAgICBAc3RhdGljbWV0aG9kCiAgICBkZWYgYmFja3dhcmQoY3R4LGR5LGR6KToKICAgICAgICBpZiBkeSBpcyBOb25lOnJldHVybiBkeixkeixOb25lCiAgICAgICAgeix3LGludmVyc2U9Y3R4LnNhdmVkX3RlbnNvcnM7ZHg9c3RhZ2VkX2R4KGR5LHosdyxpbnZlcnNlKQogICAgICAgIGlmIGR6IGlzIG5vdCBOb25lOmR4PWR4K2R6CiAgICAgICAgcmV0dXJuIGR4LGR4LE5vbmUKCmRlZiBmaXh0dXJlKHJvd3Msc2NhbGU9MS4sZGV2aWNlPSdjcHUnLGFsaWFzPSdpbmRlcGVuZGVudCcsb25seV9hPUZhbHNlKToKICAgICMgUHVyZSBkZXRlcm1pbmlzdGljIHZhbHVlczsgbm8gZ2xvYmFsIENQVS9DVURBIFJORyBjaGFuZ2UuCiAgICB2YWx1ZXM9dG9yY2guYXJhbmdlKChyb3dzKzEpKkgsZGV2aWNlPWRldmljZSxkdHlwZT10b3JjaC5mbG9hdDMyKS5yZXNoYXBlKHJvd3MrMSxIKQogICAgdmFsdWVzPSh2YWx1ZXMucmVtYWluZGVyKDk5NykvNDk5Li0xLikqc2NhbGUKICAgIGlmIGFsaWFzPT0naW5kZXBlbmRlbnQnOmE9dmFsdWVzWzpyb3dzXS5jbG9uZSgpO2I9dmFsdWVzWzE6XS5jbG9uZSgpKjAuMzcKICAgIGVsaWYgYWxpYXM9PSdzYW1lJzphPXZhbHVlc1s6cm93c10uY2xvbmUoKTtiPWEKICAgIGVsaWYgYWxpYXM9PSdvdmVybGFwJzphPXZhbHVlc1s6cm93c107Yj12YWx1ZXNbMTpdCiAgICBlbHNlOnJhaXNlIFZhbHVlRXJyb3IoYWxpYXMpCiAgICBhLnJlcXVpcmVzX2dyYWRfKFRydWUpCiAgICBpZiBub3Qgb25seV9hOmIucmVxdWlyZXNfZ3JhZF8oVHJ1ZSkKICAgIHc9MS4rdG9yY2guYXJhbmdlKEgsZGV2aWNlPWRldmljZSxkdHlwZT10b3JjaC5mbG9hdDMyKS5yZW1haW5kZXIoMTkpKjAuMDA3CiAgICByZXR1cm4gYSxiLHcKCmRlZiBlbmdpbmVlcmluZ19jbG9zZShhY3R1YWwsZXhwZWN0ZWQsKixmb3J3YXJkPUZhbHNlLHNjYWxlPTEuKToKICAgIGFzc2VydCBhY3R1YWwuc2hhcGU9PWV4cGVjdGVkLnNoYXBlIGFuZCBhY3R1YWwuZHR5cGU9PWV4cGVjdGVkLmR0eXBlPT10b3JjaC5mbG9hdDMyCiAgICBmb3IgcHJlZGljYXRlIGluICh0b3JjaC5pc25hbix0b3JjaC5pc3Bvc2luZix0b3JjaC5pc25lZ2luZik6CiAgICAgICAgYXNzZXJ0IHRvcmNoLmVxdWFsKHByZWRpY2F0ZShhY3R1YWwpLHByZWRpY2F0ZShleHBlY3RlZCkpLCdub25maW5pdGUgY2xhc3MgZGlmZmVycycKICAgIGZpbml0ZT10b3JjaC5pc2Zpbml0ZShhY3R1YWwpJnRvcmNoLmlzZmluaXRlKGV4cGVjdGVkKQogICAgYT1hY3R1YWxbZmluaXRlXS9zY2FsZTtiPWV4cGVjdGVkW2Zpbml0ZV0vc2NhbGUKICAgIGVycm9yPWZsb2F0KChhLWIpLmFicygpLm1heCgpKSBpZiBhLm51bWVsKCkgZWxzZSAwLgogICAgaWYgZm9yd2FyZDp0b3JjaC50ZXN0aW5nLmFzc2VydF9jbG9zZShhLGIscnRvbD0xZS00LGF0b2w9MWUtNSkKICAgIGVsc2U6dG9yY2gudGVzdGluZy5hc3NlcnRfY2xvc2UoYSxiLHJ0b2w9MWUtMyxhdG9sPTJlLTUpCiAgICByZXR1cm4geydtYXhfYWJzX2Vycm9yX3NjYWxlZCc6ZXJyb3IsJ3NjYWxlX2Zvcl9yZXBvcnRpbmcnOnNjYWxlLCdub25maW5pdGVfY2xhc3NfZXhhY3QnOlRydWV9CgpkZWYgY29tcGFyZShmbixyb3dzLGFjdGl2YXRpb25fc2NhbGU9MS4sdXBzdHJlYW1fc2NhbGU9MS4sZGV2aWNlPSdjcHUnLGFsaWFzPSdpbmRlcGVuZGVudCcsdXNlPSdib3RoJyxvbmx5X2E9RmFsc2UpOgogICAgYSxiLHc9Zml4dHVyZShyb3dzLGFjdGl2YXRpb25fc2NhbGUsZGV2aWNlLGFsaWFzLG9ubHlfYSkKICAgIHZlcnNpb25zPVt4Ll92ZXJzaW9uIGZvciB4IGluIChhLGIsdyldO3NuYXBzPVt4LmRldGFjaCgpLmNsb25lKCkgZm9yIHggaW4gKGEsYix3KV0KICAgICMgQSByZWFsIGNvbm5lY3RlZCBub256ZXJvIGNvdGFuZ2VudDsgbm8gY2hhbmdlcyB0byBwcm9kdWN0aW9uIHNjYWxpbmcuCiAgICBkeT10b3JjaC5saW5zcGFjZSgtLjcsLjkscm93cypILGRldmljZT1kZXZpY2UpLnJlc2hhcGUocm93cyxIKSp1cHN0cmVhbV9zY2FsZQogICAgZHo9dG9yY2gubGluc3BhY2UoLjIsLS4zLHJvd3MqSCxkZXZpY2U9ZGV2aWNlKS5yZXNoYXBlKHJvd3MsSCkqdXBzdHJlYW1fc2NhbGUKICAgIGFjdHVhbD1mbihhLGIsdyk7ZXhwZWN0ZWQ9bmF0aXZlKGEsYix3KQogICAgb3V0PVtlbmdpbmVlcmluZ19jbG9zZSh4LHksZm9yd2FyZD1UcnVlKSBmb3IgeCx5IGluIHppcChhY3R1YWwsZXhwZWN0ZWQpXQogICAgYXNzZXJ0IHRvcmNoLmVxdWFsKGFjdHVhbFsxXSxleHBlY3RlZFsxXSksJ3JvdW5kZWQgcmVzaWR1YWwgY2hhbmdlZCcKICAgIGlucz0oYSxiKSBpZiBiLnJlcXVpcmVzX2dyYWQgYW5kIGIgaXMgbm90IGEgZWxzZSAoYSwpCiAgICBzZWxlY3RlZD0oMCwxKSBpZiB1c2U9PSdib3RoJyBlbHNlICgwLCkgaWYgdXNlPT0nbm9ybV9vbmx5JyBlbHNlICgxLCkKICAgIHVwPShkeSxkeikKICAgIGdvdD10b3JjaC5hdXRvZ3JhZC5ncmFkKHR1cGxlKGFjdHVhbFtpXSBmb3IgaSBpbiBzZWxlY3RlZCksaW5zLHR1cGxlKHVwW2ldIGZvciBpIGluIHNlbGVjdGVkKSkKICAgIHdhbnQ9dG9yY2guYXV0b2dyYWQuZ3JhZCh0dXBsZShleHBlY3RlZFtpXSBmb3IgaSBpbiBzZWxlY3RlZCksaW5zLHR1cGxlKHVwW2ldIGZvciBpIGluIHNlbGVjdGVkKSkKICAgIGdyYWQ9W2VuZ2luZWVyaW5nX2Nsb3NlKHgseSxzY2FsZT11cHN0cmVhbV9zY2FsZSkgZm9yIHgseSBpbiB6aXAoZ290LHdhbnQpXQogICAgZm9yIHgsdixzIGluIHppcCgoYSxiLHcpLHZlcnNpb25zLHNuYXBzKTphc3NlcnQgeC5fdmVyc2lvbj09diBhbmQgdG9yY2guZXF1YWwoeCxzKSwnaW5wdXQgbXV0YXRlZCcKICAgIHJldHVybiB7J3Jvd3MnOnJvd3MsJ2FjdGl2YXRpb25fc2NhbGUnOmFjdGl2YXRpb25fc2NhbGUsJ3Vwc3RyZWFtX3NjYWxlJzp1cHN0cmVhbV9zY2FsZSwnYWxpYXMnOmFsaWFzLCd1c2UnOnVzZSwnb25seV9hX3JlcXVpcmVzX2dyYWQnOm9ubHlfYSwnZm9yd2FyZCc6b3V0LCdncmFkaWVudHMnOmdyYWR9CgpkZWYgZXh0cmVtZV9kaWFnbm9zdGljcyhmbixkZXZpY2U9J2NwdScpOgogICAgIyBLbm93biBleHBhbmRlZC1ncmFwaCBjYW5jZWxsYXRpb246IGRpc2NyZXBhbmN5IGlzIGRpc2Nsb3NlZCwgbm90IHByb21vdGVkLgogICAgcmVzdWx0cz1bXQogICAgZm9yIGxhYmVsLGZuMSBpbiBbKCduYXRpdmVfZXhwYW5kZWQnLG5hdGl2ZSksKCdjYW5kaWRhdGVfYXRvbWljJyxmbildOgogICAgICAgIGE9dG9yY2guZnVsbCgoMSxIKSw1MTIuLGRldmljZT1kZXZpY2UscmVxdWlyZXNfZ3JhZD1UcnVlKTtiPXRvcmNoLmZ1bGxfbGlrZShhLDUxMi4scmVxdWlyZXNfZ3JhZD1UcnVlKTt3PXRvcmNoLm9uZXMoSCxkZXZpY2U9ZGV2aWNlKQogICAgICAgIHksej1mbjEoYSxiLHcpO2c9dG9yY2guYXV0b2dyYWQuZ3JhZCgoeSx6KSwoYSwpLCAodG9yY2guZnVsbF9saWtlKHksMi4qKjM0KSx0b3JjaC5vbmVzX2xpa2UoeikpKVswXQogICAgICAgIHJlc3VsdHMuYXBwZW5kKHsnbWV0aG9kJzpsYWJlbCwnY2FzZSc6J2NhbmNlbGxhdGlvbicsJ21pbic6ZmxvYXQoZy5taW4oKSksJ21heCc6ZmxvYXQoZy5tYXgoKSksJ25hbic6aW50KGcuaXNuYW4oKS5zdW0oKSl9KQogICAgICAgIGZvciB1c2UgaW4gKCdyZXNpZHVhbF9vbmx5JywnY29ubmVjdGVkX3plcm9fbm9ybScpOgogICAgICAgICAgICBhPXRvcmNoLmZ1bGwoKDEsSCksMS41ZTM4LGRldmljZT1kZXZpY2UscmVxdWlyZXNfZ3JhZD1UcnVlKTtiPXRvcmNoLmZ1bGxfbGlrZShhLDEuNWUzOCxyZXF1aXJlc19ncmFkPVRydWUpCiAgICAgICAgICAgIHksej1mbjEoYSxiLHcpCiAgICAgICAgICAgIGc9dG9yY2guYXV0b2dyYWQuZ3JhZCgoeiwpIGlmIHVzZT09J3Jlc2lkdWFsX29ubHknIGVsc2UgKHkseiksKGEsKSwgKHRvcmNoLm9uZXNfbGlrZSh6KSwpIGlmIHVzZT09J3Jlc2lkdWFsX29ubHknIGVsc2UgKHRvcmNoLnplcm9zX2xpa2UoeSksdG9yY2gub25lc19saWtlKHopKSlbMF0KICAgICAgICAgICAgcmVzdWx0cy5hcHBlbmQoeydtZXRob2QnOmxhYmVsLCdjYXNlJzp1c2UsJ25hbic6aW50KGcuaXNuYW4oKS5zdW0oKSksJ2FsbF9vbmUnOmJvb2wodG9yY2guYWxsKGc9PTEpKX0pCiAgICByZXR1cm4gcmVzdWx0cwo=', 'solution/residual_norm.py': 'IiIiT3JpZ2luYWwgdHdvLW91dHB1dCBGUDMyIHJlc2lkdWFsLWFkZC9ub3JtIGNvbXBvbmVudDsgbm8gbW9kZWwgaG9vay4KCk9uZSBuYXR1cmFsIFF3ZW4gZGVjb2RlciBib3VuZGFyeTogeiA9IHJlc2lkdWFsICsgYXR0ZW50aW9uX291dHB1dCwgdGhlbgpub3JtKHopOyBib3RoIHJvdW5kZWQgeiBhbmQgbm9ybSBvdXRwdXQgcmVtYWluIG9ic2VydmFibGUuIE5vIGluLXBsYWNlIHVwZGF0ZSwKcHJlY2lzaW9uIG5hcnJvd2luZywgZFdlaWdodCBzdXBwcmVzc2lvbiBmb3IgdHJhaW5hYmxlIHdlaWdodHMsIG9yIGZha2Ugd29yay4KQmFja3dhcmQgcmV0YWlucyBzdGFnZWQgbm9ybSBhcml0aG1ldGljLCB0aGVuIGFkZHMgYSBnZW51aW5lIGJ5cGFzcyBjb3RhbmdlbnQuClRoaXMgYXRvbWljIGJvdW5kYXJ5IGlzIE5PVCBiaXR3aXNlIGVxdWl2YWxlbnQgdG8gYWxsIGV4cGFuZGVkIG5hdGl2ZSBncmFwaHMuCiIiIgppbXBvcnQgdG9yY2gKaW1wb3J0IHRyaXRvbgppbXBvcnQgdHJpdG9uLmxhbmd1YWdlIGFzIHRsCmZyb20gdG9yY2guYXV0b2dyYWQuZnVuY3Rpb24gaW1wb3J0IG9uY2VfZGlmZmVyZW50aWFibGUKCkB0cml0b24uaml0CmRlZiByZXNpZHVhbF9ub3JtX2YzMihBLEIsVyxaLFksUixIOnRsLmNvbnN0ZXhwcixFUFM6dGwuY29uc3RleHByLEtFRVA6dGwuY29uc3RleHByLEJMT0NLOnRsLmNvbnN0ZXhwcik6CiAgICByb3c9dGwucHJvZ3JhbV9pZCgwKTtjb2w9dGwuYXJhbmdlKDAsQkxPQ0spO3ZhbGlkPWNvbDxICiAgICBhPXRsLmxvYWQoQStyb3cqSCtjb2wsdmFsaWQsb3RoZXI9MCkudG8odGwuZmxvYXQzMikKICAgIGI9dGwubG9hZChCK3JvdypIK2NvbCx2YWxpZCxvdGhlcj0wKS50byh0bC5mbG9hdDMyKQogICAgIyBOYXRpdmUgRlAzMiBhZGQgaXMgcm91bmRlZCBiZWZvcmUgc3F1YXJlL25vcm1hbGl6YXRpb24gYW5kIGJlZm9yZSBieXBhc3MuCiAgICB6PWErYgogICAgdz10bC5sb2FkKFcrY29sLHZhbGlkLG90aGVyPTApLnRvKHRsLmZsb2F0MzIpCiAgICBtZWFuX3NxdWFyZT10bC5kaXZfcm4odGwuc3VtKHoqeiwwKSxIKQogICAgaW52ZXJzZT10bC5yc3FydChtZWFuX3NxdWFyZStFUFMpCiAgICBub3JtYWxpemVkPXoqaW52ZXJzZQogICAgeT13Km5vcm1hbGl6ZWQKICAgIHRsLnN0b3JlKForcm93KkgrY29sLHosdmFsaWQpO3RsLnN0b3JlKFkrcm93KkgrY29sLHksdmFsaWQpCiAgICBpZiBLRUVQOnRsLnN0b3JlKFIrcm93LGludmVyc2UpCgpAdHJpdG9uLmppdApkZWYgcmVzaWR1YWxfbm9ybV9hZGpvaW50X2YzMihEWSxEWixaLFcsUixEWCxIOnRsLmNvbnN0ZXhwcixIQVNfQllQQVNTOnRsLmNvbnN0ZXhwcixCTE9DSzp0bC5jb25zdGV4cHIpOgogICAgcm93PXRsLnByb2dyYW1faWQoMCk7Y29sPXRsLmFyYW5nZSgwLEJMT0NLKTt2YWxpZD1jb2w8SAogICAgej10bC5sb2FkKForcm93KkgrY29sLHZhbGlkLG90aGVyPTApLnRvKHRsLmZsb2F0MzIpCiAgICB3PXRsLmxvYWQoVytjb2wsdmFsaWQsb3RoZXI9MCkudG8odGwuZmxvYXQzMikKICAgIGR5PXRsLmxvYWQoRFkrcm93KkgrY29sLHZhbGlkLG90aGVyPTApLnRvKHRsLmZsb2F0MzIpCiAgICBpbnZlcnNlPXRsLmxvYWQoUityb3cpLnRvKHRsLmZsb2F0MzIpCiAgICBnPWR5Knc7ZGlyZWN0PWcqaW52ZXJzZQogICAgZGludmVyc2U9dGwuc3VtKGcqeiwwKQogICAgaW52ZXJzZTM9KGludmVyc2UqaW52ZXJzZSkqaW52ZXJzZQogICAgZHZhcmlhbmNlPSgtMC41KmRpbnZlcnNlKSppbnZlcnNlMwogICAgdmFyaWFuY2VfcGF0aD10bC5kaXZfcm4oZHZhcmlhbmNlLEgpKigyLjAqeikKICAgIGR4PWRpcmVjdCt2YXJpYW5jZV9wYXRoCiAgICBpZiBIQVNfQllQQVNTOmR4PWR4K3RsLmxvYWQoRForcm93KkgrY29sLHZhbGlkLG90aGVyPTApLnRvKHRsLmZsb2F0MzIpCiAgICB0bC5zdG9yZShEWCtyb3cqSCtjb2wsZHgsdmFsaWQpCgoKZGVmIHN1cHBvcnRlZChhLGIsdyxlcHMpOgogICAgcmV0dXJuICh0eXBlKGEpIGlzIHRvcmNoLlRlbnNvciBhbmQgdHlwZShiKSBpcyB0b3JjaC5UZW5zb3IKICAgICAgICBhbmQgdHlwZSh3KSBpbiAodG9yY2guVGVuc29yLHRvcmNoLm5uLlBhcmFtZXRlcikKICAgICAgICBhbmQgYS5kZXZpY2UudHlwZT09J2N1ZGEnIGFuZCBhLmRldmljZT09Yi5kZXZpY2U9PXcuZGV2aWNlCiAgICAgICAgYW5kIGEuZHR5cGU9PWIuZHR5cGU9PXcuZHR5cGU9PXRvcmNoLmZsb2F0MzIKICAgICAgICBhbmQgYS5uZGltIGluICgyLDMpIGFuZCBhLnNoYXBlPT1iLnNoYXBlIGFuZCBhLnNoYXBlWy0xXT09MTUzNgogICAgICAgIGFuZCAxPD1hLm51bWVsKCkvLzE1MzY8PTIwNDggYW5kIHcuc2hhcGU9PSgxNTM2LCkgYW5kIG5vdCB3LnJlcXVpcmVzX2dyYWQKICAgICAgICBhbmQgYWxsKHguaXNfY29udGlndW91cygpIGFuZCBub3QgeC5pc19jb25qKCkgYW5kIG5vdCB4LmlzX25lZygpIGZvciB4IGluIChhLGIsdykpCiAgICAgICAgYW5kIHR5cGUoZXBzKSBpcyBmbG9hdCBhbmQgZXBzPT0xZS02KQoKY2xhc3MgUmVzaWR1YWxOb3JtKHRvcmNoLmF1dG9ncmFkLkZ1bmN0aW9uKToKICAgIEBzdGF0aWNtZXRob2QKICAgIGRlZiBmb3J3YXJkKGN0eCxhLGIsdyxlcHMsbmVlZF9iYWNrd2FyZCk6CiAgICAgICAgcm93cz1hLm51bWVsKCkvLzE1MzY7ej10b3JjaC5lbXB0eV9saWtlKGEpO3k9dG9yY2guZW1wdHlfbGlrZShhKQogICAgICAgIHI9dG9yY2guZW1wdHkoKHJvd3MsKSBpZiBuZWVkX2JhY2t3YXJkIGVsc2UgKDAsKSxkZXZpY2U9YS5kZXZpY2UsZHR5cGU9dG9yY2guZmxvYXQzMikKICAgICAgICByZXNpZHVhbF9ub3JtX2YzMlsocm93cywpXShhLGIsdyx6LHksciwxNTM2LGVwcyxuZWVkX2JhY2t3YXJkLDIwNDgsbnVtX3dhcnBzPTQsZW5hYmxlX2ZwX2Z1c2lvbj1GYWxzZSkKICAgICAgICBjdHguc2V0X21hdGVyaWFsaXplX2dyYWRzKEZhbHNlKQogICAgICAgIGlmIG5lZWRfYmFja3dhcmQ6Y3R4LnNhdmVfZm9yX2JhY2t3YXJkKHosdyxyKQogICAgICAgIHJldHVybiB5LHoKICAgIEBzdGF0aWNtZXRob2QKICAgIEBvbmNlX2RpZmZlcmVudGlhYmxlCiAgICBkZWYgYmFja3dhcmQoY3R4LGR5LGR6KToKICAgICAgICAjIEFuIHVudXNlZCBub3JtIG91dHB1dCBtdXN0IG5vdCBleGVjdXRlIGl0cyBkZXJpdmF0aXZlIGZyb20gZmFrZSB6ZXJvcy4KICAgICAgICBpZiBkeSBpcyBOb25lOgogICAgICAgICAgICByZXR1cm4gZHogaWYgY3R4Lm5lZWRzX2lucHV0X2dyYWRbMF0gZWxzZSBOb25lLGR6IGlmIGN0eC5uZWVkc19pbnB1dF9ncmFkWzFdIGVsc2UgTm9uZSxOb25lLE5vbmUsTm9uZQogICAgICAgIHosdyxyPWN0eC5zYXZlZF90ZW5zb3JzO2R5PWR5LmNvbnRpZ3VvdXMoKQogICAgICAgIGlmIGR6IGlzIG5vdCBOb25lOmR6PWR6LmNvbnRpZ3VvdXMoKQogICAgICAgIGR4PXRvcmNoLmVtcHR5X2xpa2UoeikKICAgICAgICByZXNpZHVhbF9ub3JtX2Fkam9pbnRfZjMyWyh6Lm51bWVsKCkvLzE1MzYsKV0oZHksZHkgaWYgZHogaXMgTm9uZSBlbHNlIGR6LHosdyxyLGR4LDE1MzYsZHogaXMgbm90IE5vbmUsMjA0OCxudW1fd2FycHM9NCxlbmFibGVfZnBfZnVzaW9uPUZhbHNlKQogICAgICAgIHJldHVybiBkeCBpZiBjdHgubmVlZHNfaW5wdXRfZ3JhZFswXSBlbHNlIE5vbmUsZHggaWYgY3R4Lm5lZWRzX2lucHV0X2dyYWRbMV0gZWxzZSBOb25lLE5vbmUsTm9uZSxOb25lCgpkZWYgZnVzZWRfcmVzaWR1YWxfbm9ybShhLGIsdyxlcHM9MWUtNik6CiAgICBpZiBub3Qgc3VwcG9ydGVkKGEsYix3LGVwcyk6cmFpc2UgVmFsdWVFcnJvcignVW5zdXBwb3J0ZWQgcmVzaWR1YWwgbm9ybSBjb21wb25lbnQgY29udHJhY3QnKQogICAgbmVlZD10b3JjaC5pc19ncmFkX2VuYWJsZWQoKSBhbmQgKGEucmVxdWlyZXNfZ3JhZCBvciBiLnJlcXVpcmVzX2dyYWQpCiAgICByZXR1cm4gUmVzaWR1YWxOb3JtLmFwcGx5KGEsYix3LGVwcyxuZWVkKQo=', 'solution/web_preflight.py': 'IiIiQm91bmRlZCBjb21wb25lbnQtb25seSB0YXJnZXQgZ2F0ZTsgcHJlc2VydmVzIHBhcnRpYWwgY2FzZSByZWNvcmRzIG9uIGZhaWx1cmUuIiIiCmltcG9ydCBhcmdwYXJzZSxoYXNobGliLGluc3BlY3QsanNvbixvcyx0cmFjZWJhY2sKZnJvbSBwYXRobGliIGltcG9ydCBQYXRoCkVYUEVDVEVEX1FXRU5fQkxPQj0nYjRmZWQ5OGZhZmFlOGU0ZWUzOWJlM2I4Y2Q3NjZiMWM4NWIwNzljZCcKCmRlZiBydW4ocmVwb3J0KToKICAgIGlmIG9zLmVudmlyb24uZ2V0KCdUUklUT05fSU5URVJQUkVUJyk6cmFpc2UgUnVudGltZUVycm9yKCdJbnRlcnByZXRlciBpcyBub3QgdGFyZ2V0IGV4ZWN1dGlvbicpCiAgICBpbXBvcnQgdG9yY2gsdHJpdG9uLHRyYW5zZm9ybWVycwogICAgZnJvbSB0cmFuc2Zvcm1lcnMubW9kZWxzLnF3ZW4yIGltcG9ydCBtb2RlbGluZ19xd2VuMiBhcyBxd2VuCiAgICBmcm9tIC5yZXNpZHVhbF9ub3JtIGltcG9ydCBmdXNlZF9yZXNpZHVhbF9ub3JtLHN1cHBvcnRlZAogICAgZnJvbSAuY29udHJhY3RzIGltcG9ydCBjb21wYXJlLG5hdGl2ZSxmaXh0dXJlLGV4dHJlbWVfZGlhZ25vc3RpY3MsSAogICAgYXNzZXJ0IHRvcmNoLl9fdmVyc2lvbl9fPT0nMi4xMS4wK2N1MTI4JyBhbmQgdHJpdG9uLl9fdmVyc2lvbl9fPT0nMy42LjAnIGFuZCB0cmFuc2Zvcm1lcnMuX192ZXJzaW9uX189PSc1LjE2LjEnCiAgICBhc3NlcnQgdG9yY2guY3VkYS5kZXZpY2VfY291bnQoKT09MSBhbmQgdG9yY2guY3VkYS5nZXRfZGV2aWNlX25hbWUoMCk9PSdUZXNsYSBUNCcgYW5kIHRvcmNoLmN1ZGEuZ2V0X2RldmljZV9jYXBhYmlsaXR5KDApPT0oNyw1KQogICAgcmF3PVBhdGgoaW5zcGVjdC5nZXRmaWxlKHF3ZW4pKS5yZWFkX2J5dGVzKCk7YmxvYj1oYXNobGliLnNoYTEoYidibG9iICcrc3RyKGxlbihyYXcpKS5lbmNvZGUoKStiJ1wwJytyYXcpLmhleGRpZ2VzdCgpCiAgICBhc3NlcnQgYmxvYj09RVhQRUNURURfUVdFTl9CTE9CCiAgICByZXBvcnQudXBkYXRlKGdwdV9leGVjdXRpb25fYXR0ZW1wdGVkPVRydWUsbmF0aXZlX3NvdXJjZV9ibG9iPWJsb2Isb3JkaW5hcnk9W10sc3RydWN0dXJhbD1bXSxleHRyZW1lX2RpYWdub3N0aWNzPVtdKQogICAgIyBUaGUgbmF0aXZlIGVhZ2VyIGV4cHJlc3Npb24gaXMgY2hlY2tlZCBhZ2FpbnN0IHRoZSBhY3R1YWwgaW5zdGFsbGVkIGNsYXNzLgogICAgYSxiLHc9Zml4dHVyZSg3LGRldmljZT0nY3VkYScpO21vZHVsZT1xd2VuLlF3ZW4yUk1TTm9ybShIKS5jdWRhKCk7bW9kdWxlLndlaWdodD10b3JjaC5ubi5QYXJhbWV0ZXIodyxyZXF1aXJlc19ncmFkPUZhbHNlKQogICAgZXhwZWN0ZWQ9bW9kdWxlKGErYik7cmVmPW5hdGl2ZShhLGIsdylbMF0KICAgIGFzc2VydCB0b3JjaC5lcXVhbChleHBlY3RlZCxyZWYpCiAgICB1cD10b3JjaC5vbmVzX2xpa2UocmVmKSoxMjguCiAgICBnYT10b3JjaC5hdXRvZ3JhZC5ncmFkKGV4cGVjdGVkLChhLGIpLHVwKTtnYj10b3JjaC5hdXRvZ3JhZC5ncmFkKHJlZiwoYSxiKSx1cCkKICAgIGFzc2VydCBhbGwodG9yY2guZXF1YWwoeCx5KSBmb3IgeCx5IGluIHppcChnYSxnYikpCiAgICByZXBvcnRbJ3N0cnVjdHVyYWwnXS5hcHBlbmQoJ25hdGl2ZV9leHByZXNzaW9uX21hdGNoZXNfaW5zdGFsbGVkX3F3ZW5fZm9yd2FyZF9iYWNrd2FyZCcpCiAgICBmb3Igcm93cyBpbiAoMSw3LDIwNDgpOgogICAgICAgIGZvciBzY2FsZSBpbiAoLjAxLDEuLDEwLik6CiAgICAgICAgICAgIHJlcG9ydFsnY3VycmVudF9jYXNlJ109eydyb3dzJzpyb3dzLCdhY3RpdmF0aW9uX3NjYWxlJzpzY2FsZSwndXBzdHJlYW1fc2NhbGUnOjY1NTM2Ln0KICAgICAgICAgICAgcmVwb3J0WydvcmRpbmFyeSddLmFwcGVuZChjb21wYXJlKGZ1c2VkX3Jlc2lkdWFsX25vcm0scm93cyxzY2FsZSw2NTUzNi4sZGV2aWNlPSdjdWRhJykpCiAgICBmb3IgYWxpYXMgaW4gKCdpbmRlcGVuZGVudCcsJ3NhbWUnLCdvdmVybGFwJyk6CiAgICAgICAgZm9yIHVzZSBpbiAoJ25vcm1fb25seScsJ3Jlc2lkdWFsX29ubHknLCdib3RoJyk6CiAgICAgICAgICAgIHJlcG9ydFsnY3VycmVudF9jYXNlJ109eydhbGlhcyc6YWxpYXMsJ3VzZSc6dXNlfQogICAgICAgICAgICByZXBvcnRbJ29yZGluYXJ5J10uYXBwZW5kKGNvbXBhcmUoZnVzZWRfcmVzaWR1YWxfbm9ybSw3LDEuLDEyOC4sZGV2aWNlPSdjdWRhJyxhbGlhcz1hbGlhcyx1c2U9dXNlKSkKICAgIHJlcG9ydFsnb3JkaW5hcnknXS5hcHBlbmQoY29tcGFyZShmdXNlZF9yZXNpZHVhbF9ub3JtLDcsMS4sMS4sZGV2aWNlPSdjdWRhJyxvbmx5X2E9VHJ1ZSkpCiAgICByZXBvcnRbJ29yZGluYXJ5J10uYXBwZW5kKGNvbXBhcmUoZnVzZWRfcmVzaWR1YWxfbm9ybSw3LDAuLDEuLGRldmljZT0nY3VkYScpKQogICAgYSxiLHc9Zml4dHVyZSg3LGRldmljZT0nY3VkYScpCiAgICBhc3NlcnQgc3VwcG9ydGVkKGEsYix3LDFlLTYpCiAgICBmb3IgdHJpcGxlLGVwcyBpbiAoKChhLmhhbGYoKSxiLmhhbGYoKSx3KSwxZS02KSwoKGEsYix3LmNsb25lKCkucmVxdWlyZXNfZ3JhZF8oKSksMWUtNiksKChhWzosOjoyXSxiWzosOjoyXSx3Wzo3NjhdKSwxZS02KSwoKGEsYix3KSwxZS01KSk6CiAgICAgICAgYXNzZXJ0IG5vdCBzdXBwb3J0ZWQoKnRyaXBsZSxlcHMpCiAgICAgICAgdHJ5OmZ1c2VkX3Jlc2lkdWFsX25vcm0oKnRyaXBsZSxlcHMpCiAgICAgICAgZXhjZXB0IFZhbHVlRXJyb3I6cGFzcwogICAgICAgIGVsc2U6cmFpc2UgQXNzZXJ0aW9uRXJyb3IoJ1Vuc3VwcG9ydGVkIGNvbnRyYWN0IGFjY2VwdGVkJykKICAgIHJlcG9ydFsnc3RydWN0dXJhbCddLmFwcGVuZCgnZHR5cGVfdHJhaW5hYmxlX3dlaWdodF9sYXlvdXRfZXBzaWxvbl9uZWdhdGl2ZXNfcmVqZWN0ZWQnKQogICAgIyBTYXZlZCBvdXRwdXRzIGFuZCB3ZWlnaHQgcmV0YWluIG5hdGl2ZSB2ZXJzaW9uLWNvdW50ZXIgcHJvdGVjdGlvbi4KICAgIGZvciB0YXJnZXQgaW4gKCdyZXNpZHVhbCcsJ3dlaWdodCcpOgogICAgICAgIGEsYix3PWZpeHR1cmUoMSxkZXZpY2U9J2N1ZGEnKTt5LHo9ZnVzZWRfcmVzaWR1YWxfbm9ybShhLGIsdykKICAgICAgICB3aXRoIHRvcmNoLm5vX2dyYWQoKTooeiBpZiB0YXJnZXQ9PSdyZXNpZHVhbCcgZWxzZSB3KS5hZGRfKDEpCiAgICAgICAgdHJ5OnRvcmNoLmF1dG9ncmFkLmdyYWQoeSwoYSxiKSx0b3JjaC5vbmVzX2xpa2UoeSkpCiAgICAgICAgZXhjZXB0IFJ1bnRpbWVFcnJvciBhcyBleGM6YXNzZXJ0ICdtb2RpZmllZCBieSBhbiBpbnBsYWNlIG9wZXJhdGlvbicgaW4gc3RyKGV4YykKICAgICAgICBlbHNlOnJhaXNlIEFzc2VydGlvbkVycm9yKCdTYXZlZCAnK3RhcmdldCsnIG11dGF0aW9uIG5vdCBkZXRlY3RlZCcpCiAgICByZXBvcnRbJ3N0cnVjdHVyYWwnXS5hcHBlbmQoJ3NhdmVkX3Jlc2lkdWFsX2FuZF93ZWlnaHRfbXV0YXRpb25fcmVqZWN0ZWQnKQogICAgc2F2ZWQ9W10KICAgIGEsYix3PWZpeHR1cmUoMSxkZXZpY2U9J2N1ZGEnKQogICAgd2l0aCB0b3JjaC5ub19ncmFkKCksdG9yY2guYXV0b2dyYWQuZ3JhcGguc2F2ZWRfdGVuc29yc19ob29rcyhsYW1iZGEgeDpzYXZlZC5hcHBlbmQoeCkgb3IgeCxsYW1iZGEgeDp4KTp5LHo9ZnVzZWRfcmVzaWR1YWxfbm9ybShhLGIsdykKICAgIGFzc2VydCBub3Qgc2F2ZWQgYW5kIG5vdCB5LnJlcXVpcmVzX2dyYWQgYW5kIG5vdCB6LnJlcXVpcmVzX2dyYWQKICAgIHJlcG9ydFsnc3RydWN0dXJhbCddLmFwcGVuZCgnbm9fZ3JhZF9zYXZlc19ub3RoaW5nJykKICAgIHJlcG9ydFsnY3VycmVudF9jYXNlJ109J2V4dHJlbWVfZGlhZ25vc3RpY3Nfbm90X3VuaXZlcnNhbF9lcXVpdmFsZW5jZV9nYXRlJwogICAgcmVwb3J0WydleHRyZW1lX2RpYWdub3N0aWNzJ109ZXh0cmVtZV9kaWFnbm9zdGljcyhmdXNlZF9yZXNpZHVhbF9ub3JtLCdjdWRhJykKICAgICMgVW51c2VkIHZlcnN1cyBjb25uZWN0ZWQtemVybyBub3JtIHBhdGhzIG11c3QgcHJlc2VydmUgbm9uZmluaXRlIGNsYXNzZXMuCiAgICByb3dzPXJlcG9ydFsnZXh0cmVtZV9kaWFnbm9zdGljcyddCiAgICBmb3IgaW5kZXggaW4gKDEsNCk6YXNzZXJ0IHJvd3NbaW5kZXhdWydhbGxfb25lJ10gYW5kIHJvd3NbaW5kZXhdWyduYW4nXT09MAogICAgZm9yIGluZGV4IGluICgyLDUpOmFzc2VydCByb3dzW2luZGV4XVsnbmFuJ109PUgKICAgICMgVGhlIGNhbmNlbGxhdGlvbiB3aXRuZXNzIGlzIHJlcG9ydGVkLCBuZXZlciBjb25jZWFsZWQgb3IgdXNlZCBhcyBhbgogICAgIyBhc3NlcnRpb24gb2YgZnVsbCBuYXRpdmUtZ3JhcGggZXF1aXZhbGVuY2UuIE5vIHByb2R1Y3Rpb24gaW50ZWdyYXRpb24geWV0LgogICAgdG9yY2guY3VkYS5zeW5jaHJvbml6ZSgpO3JlcG9ydFsnZ3B1X2V4ZWN1dGVkJ109VHJ1ZQoKZGVmIG1haW4oKToKICAgIHA9YXJncGFyc2UuQXJndW1lbnRQYXJzZXIoKTtwLmFkZF9hcmd1bWVudCgnLS1vdXQnLHJlcXVpcmVkPVRydWUpO2FyZ3M9cC5wYXJzZV9hcmdzKCkKICAgIHJlcG9ydD17J3N0YXR1cyc6J0ZBSUxFRF9QUkVGTElHSFRfT05MWScsJ3Njb3BlJzonTUlDUk9CRU5DSF9PTkxZJywndHJhaW5pbmdfZXhlY3V0ZWQnOkZhbHNlLCdjb21wbGV0ZV9maXhlZF90YXNrJzpGYWxzZSwnc2NvcmUnOk5vbmUsJ2FkbWlzc2lvbic6J1VOS05PV04nLCdncHVfZXhlY3V0ZWQnOkZhbHNlfQogICAgdHJ5OnJ1bihyZXBvcnQpO3JlcG9ydFsnc3RhdHVzJ109J1BBU1NFRF9QUkVGTElHSFRfT05MWScKICAgIGV4Y2VwdCBCYXNlRXhjZXB0aW9uIGFzIGU6cmVwb3J0LnVwZGF0ZShlcnJvcj1yZXByKGUpLHRyYWNlYmFjaz10cmFjZWJhY2suZm9ybWF0X2V4YygpKQogICAgUGF0aChhcmdzLm91dCkud3JpdGVfdGV4dChqc29uLmR1bXBzKHJlcG9ydCxpbmRlbnQ9MixhbGxvd19uYW49RmFsc2UpKydcbicpCiAgICBwcmludChqc29uLmR1bXBzKHsnc3RhdHVzJzpyZXBvcnRbJ3N0YXR1cyddLCdvcmRpbmFyeV9jYXNlcyc6bGVuKHJlcG9ydC5nZXQoJ29yZGluYXJ5JyxbXSkpLCdvdXRwdXQnOmFyZ3Mub3V0fSkpCiAgICByYWlzZSBTeXN0ZW1FeGl0KHJlcG9ydFsnc3RhdHVzJ10hPSdQQVNTRURfUFJFRkxJR0hUX09OTFknKQppZiBfX25hbWVfXz09J19fbWFpbl9fJzptYWluKCkK', 'old_solution/__init__.py': 'IiIiRnJvemVuIGhpc3RvcmljYWwgb3JpZ2luYWwgUk1TTm9ybSBjb21wYXJhdG9yOyBubyBhdXRvbWF0aWMgaW5zdGFsbGF0aW9uLiIiIgo=', 'old_solution/rmsnorm.py': 'IiIiT3JpZ2luYWwgRlAzMiBmcm96ZW4tYWZmaW5lIFJNU05vcm0gdHJpYWw7IG5vIHJlc2lkdWFsIGZ1c2lvbiBvciBib3Jyb3dlZCBrZXJuZWwuCgpUaGUgZGVyaXZhdGl2ZSByZXRhaW5zIHRoZSBzb3VyY2UgZ3JhcGgncyBpbmRpdmlkdWFsIEZQMzIgYXJpdGhtZXRpYyBzdGFnZXMuCkl0IGlzIG5vdCBhIGNsYWltIG9mIGJpdHdpc2UgZXF1aXZhbGVuY2UgdG8gZXZlcnkgcGFyYWxsZWwgcmVkdWN0aW9uIG9yIGV4dGVybmFsCmF1dG9ncmFkIGFjY3VtdWxhdGlvbiBvcmRlci4gVW5zdXBwb3J0ZWQgbW9kdWxlIGNvbnRyYWN0cyB1c2UgdGhlIG9yaWdpbmFsIG1ldGhvZC4KIiIiCmltcG9ydCB0b3JjaAppbXBvcnQgdHJpdG9uCmltcG9ydCB0cml0b24ubGFuZ3VhZ2UgYXMgdGwKZnJvbSB0b3JjaC5hdXRvZ3JhZC5mdW5jdGlvbiBpbXBvcnQgb25jZV9kaWZmZXJlbnRpYWJsZQoKCkB0cml0b24uaml0CmRlZiBzb3VwX3JtczMyX2ZvcndhcmQoWCwgVywgWSwgUiwgSDogdGwuY29uc3RleHByLCBFUFM6IHRsLmNvbnN0ZXhwciwKICAgICAgICAgICAgICAgICAgICAgICBLRUVQX1I6IHRsLmNvbnN0ZXhwciwgQkxPQ0s6IHRsLmNvbnN0ZXhwcik6CiAgICByb3cgPSB0bC5wcm9ncmFtX2lkKDApCiAgICBjb2wgPSB0bC5hcmFuZ2UoMCwgQkxPQ0spCiAgICB2YWxpZCA9IGNvbCA8IEgKICAgIHggPSB0bC5sb2FkKFggKyByb3cgKiBIICsgY29sLCB2YWxpZCwgb3RoZXI9MCkudG8odGwuZmxvYXQzMikKICAgIHcgPSB0bC5sb2FkKFcgKyBjb2wsIHZhbGlkLCBvdGhlcj0wKS50byh0bC5mbG9hdDMyKQogICAgc3F1YXJlcyA9IHggKiB4CiAgICB2YXJpYW5jZSA9IHRsLmRpdl9ybih0bC5zdW0oc3F1YXJlcywgYXhpcz0wKSwgSCkKICAgIGludiA9IHRsLnJzcXJ0KHZhcmlhbmNlICsgRVBTKQogICAgbm9ybWFsaXplZCA9IHggKiBpbnYKICAgIHkgPSB3ICogbm9ybWFsaXplZAogICAgdGwuc3RvcmUoWSArIHJvdyAqIEggKyBjb2wsIHksIHZhbGlkKQogICAgaWYgS0VFUF9SOgogICAgICAgIHRsLnN0b3JlKFIgKyByb3csIGludikKCgpAdHJpdG9uLmppdApkZWYgc291cF9ybXMzMl9iYWNrd2FyZChEWSwgWCwgVywgUiwgRFgsIEg6IHRsLmNvbnN0ZXhwciwgQkxPQ0s6IHRsLmNvbnN0ZXhwcik6CiAgICByb3cgPSB0bC5wcm9ncmFtX2lkKDApCiAgICBjb2wgPSB0bC5hcmFuZ2UoMCwgQkxPQ0spCiAgICB2YWxpZCA9IGNvbCA8IEgKICAgIHggPSB0bC5sb2FkKFggKyByb3cgKiBIICsgY29sLCB2YWxpZCwgb3RoZXI9MCkudG8odGwuZmxvYXQzMikKICAgIHcgPSB0bC5sb2FkKFcgKyBjb2wsIHZhbGlkLCBvdGhlcj0wKS50byh0bC5mbG9hdDMyKQogICAgZHkgPSB0bC5sb2FkKERZICsgcm93ICogSCArIGNvbCwgdmFsaWQsIG90aGVyPTApLnRvKHRsLmZsb2F0MzIpCiAgICBpbnYgPSB0bC5sb2FkKFIgKyByb3cpLnRvKHRsLmZsb2F0MzIpCiAgICBnID0gZHkgKiB3CiAgICBkaXJlY3QgPSBnICogaW52CiAgICBkciA9IHRsLnN1bShnICogeCwgYXhpcz0wKQogICAgaW52X2N1YmVkID0gKGludiAqIGludikgKiBpbnYKICAgIGR2ID0gKC0wLjUgKiBkcikgKiBpbnZfY3ViZWQKICAgIHNjYWxlZF92YXJpYW5jZSA9IHRsLmRpdl9ybihkdiwgSCkKICAgIHZhcmlhbmNlX3BhdGggPSBzY2FsZWRfdmFyaWFuY2UgKiAoMi4wICogeCkKICAgIGR4ID0gZGlyZWN0ICsgdmFyaWFuY2VfcGF0aAogICAgdGwuc3RvcmUoRFggKyByb3cgKiBIICsgY29sLCBkeCwgdmFsaWQpCgoKY2xhc3MgRnJvemVuUk1TMzIodG9yY2guYXV0b2dyYWQuRnVuY3Rpb24pOgogICAgQHN0YXRpY21ldGhvZAogICAgZGVmIGZvcndhcmQoY3R4LCB4LCB3ZWlnaHQsIGVwcywgbmVlZF9iYWNrd2FyZCk6CiAgICAgICAgIyBUaGUgcHVibGljIGhlbHBlciBjaGVja3MgdGhlIGNvbnRyYWN0OyBubyBpbnB1dCwgZ3JhZGllbnQgb3Igd2VpZ2h0IG11dGF0aW9uLgogICAgICAgIGggPSB4LnNoYXBlWy0xXQogICAgICAgIHJvd3MgPSB4Lm51bWVsKCkgLy8gaAogICAgICAgIHkgPSB0b3JjaC5lbXB0eV9saWtlKHgpCiAgICAgICAga2VlcCA9IGJvb2wobmVlZF9iYWNrd2FyZCkKICAgICAgICBpbnYgPSB0b3JjaC5lbXB0eSgocm93cywpIGlmIGtlZXAgZWxzZSAoMCwpLCBkZXZpY2U9eC5kZXZpY2UsIGR0eXBlPXRvcmNoLmZsb2F0MzIpCiAgICAgICAgaWYgcm93czoKICAgICAgICAgICAgc291cF9ybXMzMl9mb3J3YXJkWyhyb3dzLCldKAogICAgICAgICAgICAgICAgeCwgd2VpZ2h0LCB5LCBpbnYsIGgsIGZsb2F0KGVwcyksIGtlZXAsCiAgICAgICAgICAgICAgICBCTE9DSz10cml0b24ubmV4dF9wb3dlcl9vZl8yKGgpLCBudW1fd2FycHM9NCwgZW5hYmxlX2ZwX2Z1c2lvbj1GYWxzZSkKICAgICAgICBpZiBrZWVwOgogICAgICAgICAgICBjdHguc2F2ZV9mb3JfYmFja3dhcmQoeCwgd2VpZ2h0LCBpbnYpCiAgICAgICAgY3R4LmggPSBoCiAgICAgICAgcmV0dXJuIHkKCiAgICBAc3RhdGljbWV0aG9kCiAgICBAb25jZV9kaWZmZXJlbnRpYWJsZQogICAgZGVmIGJhY2t3YXJkKGN0eCwgZHkpOgogICAgICAgIHgsIHdlaWdodCwgaW52ID0gY3R4LnNhdmVkX3RlbnNvcnMKICAgICAgICBkeSA9IGR5LmNvbnRpZ3VvdXMoKQogICAgICAgIGR4ID0gdG9yY2guZW1wdHlfbGlrZSh4KQogICAgICAgIHJvd3MgPSB4Lm51bWVsKCkgLy8gY3R4LmgKICAgICAgICBpZiByb3dzOgogICAgICAgICAgICBzb3VwX3JtczMyX2JhY2t3YXJkWyhyb3dzLCldKAogICAgICAgICAgICAgICAgZHksIHgsIHdlaWdodCwgaW52LCBkeCwgY3R4LmgsCiAgICAgICAgICAgICAgICBCTE9DSz10cml0b24ubmV4dF9wb3dlcl9vZl8yKGN0eC5oKSwgbnVtX3dhcnBzPTQsIGVuYWJsZV9mcF9mdXNpb249RmFsc2UpCiAgICAgICAgcmV0dXJuIGR4LCBOb25lLCBOb25lLCBOb25lCgoKZGVmIHN1cHBvcnRlZCh4LCB3ZWlnaHQpOgogICAgcmV0dXJuIChpc2luc3RhbmNlKHgsIHRvcmNoLlRlbnNvcikgYW5kIGlzaW5zdGFuY2Uod2VpZ2h0LCB0b3JjaC5UZW5zb3IpCiAgICAgICAgICAgIGFuZCB4LmlzX2N1ZGEgYW5kIHdlaWdodC5pc19jdWRhIGFuZCB4LmRldmljZSA9PSB3ZWlnaHQuZGV2aWNlCiAgICAgICAgICAgIGFuZCB4LmR0eXBlID09IHRvcmNoLmZsb2F0MzIgYW5kIHdlaWdodC5kdHlwZSA9PSB0b3JjaC5mbG9hdDMyCiAgICAgICAgICAgIGFuZCB4Lm5kaW0gPj0gMSBhbmQgMCA8IHguc2hhcGVbLTFdIDw9IDQwOTYKICAgICAgICAgICAgYW5kIHdlaWdodC5zaGFwZSA9PSAoeC5zaGFwZVstMV0sKSBhbmQgbm90IHdlaWdodC5yZXF1aXJlc19ncmFkCiAgICAgICAgICAgIGFuZCB4LmlzX2NvbnRpZ3VvdXMoKSBhbmQgd2VpZ2h0LmlzX2NvbnRpZ3VvdXMoKSkKCgpkZWYgZnJvemVuX3JtczMyKHgsIHdlaWdodCwgZXBzPTFlLTYpOgogICAgaWYgbm90IHN1cHBvcnRlZCh4LCB3ZWlnaHQpOgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoJ2Zyb3plbl9ybXMzMiBuZWVkcyBjb250aWd1b3VzIENVREEgRlAzMiB4IGFuZCBmcm96ZW4gRlAzMiB3ZWlnaHQnKQogICAgbmVlZF9iYWNrd2FyZCA9IHRvcmNoLmlzX2dyYWRfZW5hYmxlZCgpIGFuZCB4LnJlcXVpcmVzX2dyYWQKICAgIHJldHVybiBGcm96ZW5STVMzMi5hcHBseSh4LCB3ZWlnaHQsIGVwcywgbmVlZF9iYWNrd2FyZCkKCgpkZWYgcGF0Y2hfcXdlbjJfbm9ybSgpOgogICAgZnJvbSB0cmFuc2Zvcm1lcnMubW9kZWxzLnF3ZW4yIGltcG9ydCBtb2RlbGluZ19xd2VuMgogICAgY2xzID0gbW9kZWxpbmdfcXdlbjIuUXdlbjJSTVNOb3JtCiAgICBvcmlnaW5hbCA9IGNscy5mb3J3YXJkCiAgICBpZiBnZXRhdHRyKG9yaWdpbmFsLCAnX3NvdXBfb3JpZ2luYWxfcm1zX3RyaWFsMDAxJywgRmFsc2UpOgogICAgICAgIHJldHVybgogICAgaWYgKGNscy5fX21vZHVsZV9fICE9ICd0cmFuc2Zvcm1lcnMubW9kZWxzLnF3ZW4yLm1vZGVsaW5nX3F3ZW4yJwogICAgICAgICAgICBvciBnZXRhdHRyKG9yaWdpbmFsLCAnX19tb2R1bGVfXycsIE5vbmUpICE9IGNscy5fX21vZHVsZV9fKToKICAgICAgICByZXR1cm4KCiAgICBkZWYgZm9yd2FyZChzZWxmLCBoaWRkZW5fc3RhdGVzKToKICAgICAgICB3ZWlnaHQgPSBnZXRhdHRyKHNlbGYsICd3ZWlnaHQnLCBOb25lKQogICAgICAgIGlmICh0eXBlKHNlbGYpIGlzIGNscyBhbmQgc3VwcG9ydGVkKGhpZGRlbl9zdGF0ZXMsIHdlaWdodCkgYW5kIGhpZGRlbl9zdGF0ZXMuc2hhcGVbLTFdID09IDE1MzYKICAgICAgICAgICAgICAgIGFuZCBpc2luc3RhbmNlKGdldGF0dHIoc2VsZiwgJ3ZhcmlhbmNlX2Vwc2lsb24nLCBOb25lKSwgKGludCwgZmxvYXQpKSk6CiAgICAgICAgICAgIHJldHVybiBmcm96ZW5fcm1zMzIoaGlkZGVuX3N0YXRlcywgd2VpZ2h0LCBzZWxmLnZhcmlhbmNlX2Vwc2lsb24pCiAgICAgICAgcmV0dXJuIG9yaWdpbmFsKHNlbGYsIGhpZGRlbl9zdGF0ZXMpCgogICAgZm9yd2FyZC5fc291cF9vcmlnaW5hbF9ybXNfdHJpYWwwMDEgPSBUcnVlCiAgICBmb3J3YXJkLl9zb3VwX3VucGF0Y2hlZF9mb3J3YXJkID0gb3JpZ2luYWwKICAgIGNscy5mb3J3YXJkID0gZm9yd2FyZAo=', 'process_guard.py': 'IiIiSXNvbGF0ZWQgTGludXggc3VicmVhcGVyIGd1YXJkaWFuLiBPd25zIGNoaWxkcmVuIGFuZCBob2xkcyB0aGUgR1BVIGxvY2suCgpUaGUgZ3VhcmRpYW4gd2F0Y2hlcyBhIHBhcmVudC1saXZlbmVzcyBwaXBlOiBkcml2ZXIgU0lHS0lMTCBvciBvdGhlciBhYnJ1cHQKZGVhdGggY2xvc2VzIHRoZSBwaXBlIGFuZCB0cmlnZ2VycyBxdWFyYW50aW5lZCBjbGVhbnVwLiBQSURGRCBzaWduYWxpbmcgaW4gdGhlCnNoYXJlZCB0YXNrIGhlbHBlciBwcmV2ZW50cyBQSUQgcmV1c2UgcmFjZXM7IGRlc2NlbmRhbnRzIGFyZSBvbmx5IHRoaXMgZGVkaWNhdGVkCmd1YXJkaWFuJ3MgY2hpbGRyZW4sIGluY2x1ZGluZyBuZXdseSBhZG9wdGVkIG9ycGhhbnMgZnJvbSBuZXN0ZWQgc2Vzc2lvbnMuClRoaXMgaXMgdGFzayBvcmNoZXN0cmF0aW9uLCBuZXZlciBzb2x1dGlvbi8gd29yayBvciBrZXJuZWwgcGFydGljaXBhdGlvbi4KIiIiCmltcG9ydCBqc29uCmltcG9ydCBvcwpmcm9tIHBhdGhsaWIgaW1wb3J0IFBhdGgKaW1wb3J0IHNlbGVjdAppbXBvcnQgc2lnbmFsCmltcG9ydCBzdWJwcm9jZXNzCmltcG9ydCBzeXMKaW1wb3J0IHRpbWUKCmltcG9ydCBvd25lZF9wcm9jZXNzX3RyZWUgYXMgdHJlZQoKCmRlZiBxdWFyYW50aW5lKHBhdGgsIHJlYXNvbik6CiAgICAiIiJGaXJzdCBmYWlsdXJlIHdpbnM7IG5ldmVyIG92ZXJ3cml0ZSBvciBhdXRvbWF0aWNhbGx5IGNsZWFyIGEgbWFya2VyLiIiIgogICAgdHJ5OgogICAgICAgIGRlc2NyaXB0b3IgPSBvcy5vcGVuKHBhdGgsIG9zLk9fV1JPTkxZIHwgb3MuT19DUkVBVCB8IG9zLk9fRVhDTCB8IG9zLk9fTk9GT0xMT1csIDBvNjAwKQogICAgZXhjZXB0IEZpbGVFeGlzdHNFcnJvcjoKICAgICAgICByZXR1cm4KICAgIHdpdGggb3MuZmRvcGVuKGRlc2NyaXB0b3IsICJ3IikgYXMgc3RyZWFtOgogICAgICAgIGpzb24uZHVtcCh7InN0YXR1cyI6ICJPUEVSQVRPUl9SRUNPTkNJTElBVElPTl9SRVFVSVJFRCIsICJyZWFzb24iOiByZWFzb24sCiAgICAgICAgICAgICAgICAgICAiZ3VhcmRpYW5fcGlkIjogb3MuZ2V0cGlkKCksICJjcmVhdGVkX3VuaXgiOiB0aW1lLnRpbWUoKX0sIHN0cmVhbSkKICAgICAgICBzdHJlYW0ud3JpdGUoIlxuIikKICAgICAgICBzdHJlYW0uZmx1c2goKQogICAgICAgIG9zLmZzeW5jKHN0cmVhbS5maWxlbm8oKSkKCgpkZWYgX3dyaXRlX3JlY2VpcHQocGF0aCwgdmFsdWUpOgogICAgcGF0aCA9IFBhdGgocGF0aCkKICAgIHBlbmRpbmcgPSBwYXRoLndpdGhfbmFtZShwYXRoLm5hbWUgKyAiLnBlbmRpbmciKQogICAgcGVuZGluZy53cml0ZV90ZXh0KGpzb24uZHVtcHModmFsdWUsIGluZGVudD0yKSArICJcbiIpCiAgICBwZW5kaW5nLnJlcGxhY2UocGF0aCkKCgpkZWYgZ3VhcmRpYW5fbWFpbigpOgogICAgY29uZmlnID0ganNvbi5sb2FkcyhzeXMuc3RkaW4ucmVhZGxpbmUoKSkKICAgIHJlYWRfZmQsIGxvY2tfZmQgPSBjb25maWdbInBhcmVudF9waXBlX2ZkIl0sIGNvbmZpZ1sibG9ja19mZCJdCiAgICBzdGFydGVkLCByZXF1ZXN0ZWRfc2lnbmFsLCBwcm9jZXNzID0gdGltZS5tb25vdG9uaWMoKSwgW05vbmVdLCBOb25lCiAgICByZXN1bHQgPSB7ImV4aXRfY29kZSI6IDEyNSwgInJvb3RfZXhpdF9jb2RlIjogTm9uZSwgInRpbWVkX291dCI6IEZhbHNlLAogICAgICAgICAgICAgICJpbnRlcnJ1cHRlZCI6IEZhbHNlLCAiY2xlYW51cF9jb25maXJtZWQiOiBGYWxzZSwKICAgICAgICAgICAgICAicmVhc29uIjogTm9uZSwgImd1YXJkaWFuX3BpZCI6IG9zLmdldHBpZCgpfQogICAgZGVmIHJlcXVlc3Qoc2lnbnVtLCBmcmFtZSk6CiAgICAgICAgcmVxdWVzdGVkX3NpZ25hbFswXSA9IHNpZ251bQogICAgZm9yIHNpZyBpbiAoc2lnbmFsLlNJR0lOVCwgc2lnbmFsLlNJR1RFUk0pOgogICAgICAgIHNpZ25hbC5zaWduYWwoc2lnLCByZXF1ZXN0KQogICAgdHJ5OgogICAgICAgIHRyZWUucHJlcGFyZSgpICAjIHJlcXVpcmVkIGJlZm9yZSBhbnkgY2hpbGQ7IG5ldmVyIHNpbGVudGx5IGRlZ3JhZGUKICAgICAgICB3aXRoIFBhdGgoY29uZmlnWyJsb2ciXSkub3BlbigieCIpIGFzIHN0cmVhbToKICAgICAgICAgICAgcHJvY2VzcyA9IHN1YnByb2Nlc3MuUG9wZW4oY29uZmlnWyJjb21tYW5kIl0sIGN3ZD1jb25maWdbImN3ZCJdLAogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICBzdGRvdXQ9c3RyZWFtLCBzdGRlcnI9c3VicHJvY2Vzcy5TVERPVVQsCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIHN0YXJ0X25ld19zZXNzaW9uPVRydWUpCiAgICAgICAgICAgIHdoaWxlIFRydWU6CiAgICAgICAgICAgICAgICByZWFkeSwgXywgXyA9IHNlbGVjdC5zZWxlY3QoW3JlYWRfZmRdLCBbXSwgW10sIDAuMDUpCiAgICAgICAgICAgICAgICBpZiByZWFkeSBhbmQgb3MucmVhZChyZWFkX2ZkLCAxKSA9PSBiIiI6CiAgICAgICAgICAgICAgICAgICAgcmVzdWx0LnVwZGF0ZShyZWFzb249InBhcmVudF9saXZlbmVzc19waXBlX2Nsb3NlZCIsIGludGVycnVwdGVkPVRydWUpCiAgICAgICAgICAgICAgICAgICAgYnJlYWsKICAgICAgICAgICAgICAgIGlmIHJlcXVlc3RlZF9zaWduYWxbMF0gaXMgbm90IE5vbmU6CiAgICAgICAgICAgICAgICAgICAgcmVzdWx0LnVwZGF0ZShyZWFzb249Imd1YXJkaWFuX3NpZ25hbF8iICsgc3RyKHJlcXVlc3RlZF9zaWduYWxbMF0pLCBpbnRlcnJ1cHRlZD1UcnVlKQogICAgICAgICAgICAgICAgICAgIGJyZWFrCiAgICAgICAgICAgICAgICBpZiB0aW1lLm1vbm90b25pYygpIC0gc3RhcnRlZCA+PSBjb25maWdbInRpbWVvdXQiXToKICAgICAgICAgICAgICAgICAgICByZXN1bHQudXBkYXRlKHJlYXNvbj0ib3V0ZXJfdGltZW91dCIsIHRpbWVkX291dD1UcnVlKQogICAgICAgICAgICAgICAgICAgIGJyZWFrCiAgICAgICAgICAgICAgICByZXR1cm5jb2RlID0gcHJvY2Vzcy5wb2xsKCkKICAgICAgICAgICAgICAgIGlmIHJldHVybmNvZGUgaXMgbm90IE5vbmU6CiAgICAgICAgICAgICAgICAgICAgcmVzdWx0WyJyb290X2V4aXRfY29kZSJdID0gcmV0dXJuY29kZQogICAgICAgICAgICAgICAgICAgIGlmIHRyZWUuZGVzY2VuZGFudHMoKToKICAgICAgICAgICAgICAgICAgICAgICAgcmVzdWx0WyJyZWFzb24iXSA9ICJyb290X2V4aXRlZF93aXRoX293bmVkX2Rlc2NlbmRhbnRzIgogICAgICAgICAgICAgICAgICAgIGVsc2U6CiAgICAgICAgICAgICAgICAgICAgICAgIHJlc3VsdC51cGRhdGUoZXhpdF9jb2RlPXJldHVybmNvZGUsIGNsZWFudXBfY29uZmlybWVkPVRydWUpCiAgICAgICAgICAgICAgICAgICAgYnJlYWsKICAgICAgICAgICAgaWYgcmVzdWx0WyJyZWFzb24iXSBpcyBub3QgTm9uZToKICAgICAgICAgICAgICAgIHF1YXJhbnRpbmUoY29uZmlnWyJtYXJrZXIiXSwgcmVzdWx0WyJyZWFzb24iXSkKICAgICAgICAgICAgICAgICMgY2xlYW51cCBpZ25vcmVzIHJlcGVhdGVkIElOVC9URVJNLCBzdG9wcy9yZXNjYW5zL2tpbGxzIGJ5CiAgICAgICAgICAgICAgICAjIHBpZGZkIGFuZCB3YWl0cy9yZWFwcyB3aXRob3V0IGFueSB0aW1lb3V0LXVubG9jayBlc2NhcGUuCiAgICAgICAgICAgICAgICB0cmVlLmNsZWFudXAocHJvY2VzcykKICAgICAgICAgICAgICAgIHJlc3VsdC51cGRhdGUocm9vdF9leGl0X2NvZGU9cHJvY2Vzcy5yZXR1cm5jb2RlLCBjbGVhbnVwX2NvbmZpcm1lZD1UcnVlKQogICAgICAgICAgICAgICAgaWYgcmVzdWx0WyJ0aW1lZF9vdXQiXToKICAgICAgICAgICAgICAgICAgICByZXN1bHRbImV4aXRfY29kZSJdID0gMTI0CiAgICAgICAgICAgICAgICBlbGlmIHJlc3VsdFsiaW50ZXJydXB0ZWQiXToKICAgICAgICAgICAgICAgICAgICByZXN1bHRbImV4aXRfY29kZSJdID0gMTMwCiAgICBleGNlcHQgQmFzZUV4Y2VwdGlvbiBhcyBleGM6CiAgICAgICAgcmVzdWx0LnVwZGF0ZShyZWFzb249Imd1YXJkaWFuX2V4Y2VwdGlvbjoiICsgcmVwcihleGMpLCBleGl0X2NvZGU9MTI1KQogICAgICAgIHRyeToKICAgICAgICAgICAgcXVhcmFudGluZShjb25maWdbIm1hcmtlciJdLCByZXN1bHRbInJlYXNvbiJdKQogICAgICAgIGZpbmFsbHk6CiAgICAgICAgICAgIGlmIHByb2Nlc3MgaXMgbm90IE5vbmU6CiAgICAgICAgICAgICAgICB0cmVlLmNsZWFudXAocHJvY2VzcykKICAgICAgICAgICAgcmVzdWx0WyJjbGVhbnVwX2NvbmZpcm1lZCJdID0gbm90IHRyZWUuZGVzY2VuZGFudHMoKQogICAgZmluYWxseToKICAgICAgICByZXN1bHRbIm91dGVyX3dhbGxfcyJdID0gdGltZS5tb25vdG9uaWMoKSAtIHN0YXJ0ZWQKICAgICAgICBfd3JpdGVfcmVjZWlwdChjb25maWdbInJlY2VpcHQiXSwgcmVzdWx0KQogICAgICAgIG9zLmNsb3NlKHJlYWRfZmQpCiAgICAgICAgIyBUaGUgaW5oZXJpdGVkIG9wZW4tZmlsZSBkZXNjcmlwdGlvbiByZXRhaW5zIHRoZSBjYW5vbmljYWwgZmxvY2sgZXZlbgogICAgICAgICMgaWYgdGhlIGRyaXZlciBoYXMgZGllZC4gQ2xvc2UgT05MWSBhZnRlciBvd25lZC10cmVlIGNsZWFudXAgY29tcGxldGVzLgogICAgICAgIG9zLmNsb3NlKGxvY2tfZmQpCiAgICByZXR1cm4gcmVzdWx0WyJleGl0X2NvZGUiXQoKCmRlZiBydW4oY29tbWFuZCwgKiwgY3dkLCBlbnYsIGxvZywgdGltZW91dCwgbG9ja19mZCwKICAgICAgICBtYXJrZXI9Ii90bXAvc291cC1ncHUwLmxvY2suYmxvY2tlZCIpOgogICAgIiIiRGVkaWNhdGVkIGRyaXZlciBBUEkuIEFueSBCYXNlRXhjZXB0aW9uIGFza3MgZ3VhcmRpYW4gY2xlYW51cCBhbmQgd2FpdHMuIiIiCiAgICByZWNlaXB0ID0gUGF0aChzdHIobG9nKSArICIucHJvY2Vzcy5qc29uIikucmVzb2x2ZSgpCiAgICBpZiByZWNlaXB0LmV4aXN0cygpOgogICAgICAgIHJhaXNlIFJ1bnRpbWVFcnJvcigiUHJvY2VzcyByZWNlaXB0IGFscmVhZHkgZXhpc3RzOyByZWZ1c2UgdG8gb3ZlcndyaXRlIikKICAgIHJlYWRfZmQsIHdyaXRlX2ZkID0gb3MucGlwZSgpCiAgICBjb25maWcgPSB7ImNvbW1hbmQiOiBbc3RyKHgpIGZvciB4IGluIGNvbW1hbmRdLCAiY3dkIjogc3RyKFBhdGgoY3dkKS5yZXNvbHZlKCkpLAogICAgICAgICAgICAgICJsb2ciOiBzdHIoUGF0aChsb2cpLnJlc29sdmUoKSksICJ0aW1lb3V0IjogdGltZW91dCwKICAgICAgICAgICAgICAicmVjZWlwdCI6IHN0cihyZWNlaXB0KSwgInBhcmVudF9waXBlX2ZkIjogcmVhZF9mZCwKICAgICAgICAgICAgICAibG9ja19mZCI6IGxvY2tfZmQsICJtYXJrZXIiOiBzdHIobWFya2VyKX0KICAgIGd1YXJkaWFuID0gTm9uZQogICAgaW50ZXJydXB0ZWQgPSBOb25lCiAgICBkZWYgcmVxdWVzdChzaWdudW0sIGZyYW1lKToKICAgICAgICByYWlzZSBJbnRlcnJ1cHRlZEVycm9yKCJEcml2ZXIgcmVjZWl2ZWQgc2lnbmFsICIgKyBzdHIoc2lnbnVtKSkKICAgIG9sZF9oYW5kbGVycyA9IHtzaWc6IHNpZ25hbC5zaWduYWwoc2lnLCByZXF1ZXN0KSBmb3Igc2lnIGluIChzaWduYWwuU0lHSU5ULCBzaWduYWwuU0lHVEVSTSl9CiAgICB0cnk6CiAgICAgICAgZ3VhcmRpYW4gPSBzdWJwcm9jZXNzLlBvcGVuKFtzeXMuZXhlY3V0YWJsZSwgc3RyKFBhdGgoX19maWxlX18pLnJlc29sdmUoKSksICItLWd1YXJkaWFuIl0sCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIHN0ZGluPXN1YnByb2Nlc3MuUElQRSwgZW52PWVudiwKICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgcGFzc19mZHM9KHJlYWRfZmQsIGxvY2tfZmQpLCBzdGFydF9uZXdfc2Vzc2lvbj1UcnVlLAogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICB0ZXh0PVRydWUpCiAgICAgICAgb3MuY2xvc2UocmVhZF9mZCkKICAgICAgICByZWFkX2ZkID0gTm9uZQogICAgICAgIGd1YXJkaWFuLnN0ZGluLndyaXRlKGpzb24uZHVtcHMoY29uZmlnKSArICJcbiIpCiAgICAgICAgZ3VhcmRpYW4uc3RkaW4uY2xvc2UoKQogICAgICAgIGd1YXJkaWFuLndhaXQoKQogICAgZXhjZXB0IEJhc2VFeGNlcHRpb24gYXMgZXhjOgogICAgICAgIGludGVycnVwdGVkID0gcmVwcihleGMpCiAgICAgICAgIyBJZ25vcmUgcmVwZWF0ZWQgc3RvcCBzaWduYWxzIHVudGlsIHRoZSBndWFyZGlhbiBjb25maXJtcyBjbGVhbnVwLgogICAgICAgIGZvciBzaWcgaW4gb2xkX2hhbmRsZXJzOgogICAgICAgICAgICBzaWduYWwuc2lnbmFsKHNpZywgc2lnbmFsLlNJR19JR04pCiAgICAgICAgaWYgd3JpdGVfZmQgaXMgbm90IE5vbmU6CiAgICAgICAgICAgIG9zLmNsb3NlKHdyaXRlX2ZkKQogICAgICAgICAgICB3cml0ZV9mZCA9IE5vbmUKICAgICAgICBpZiBndWFyZGlhbiBpcyBub3QgTm9uZToKICAgICAgICAgICAgaWYgZ3VhcmRpYW4uc3RkaW4gaXMgbm90IE5vbmUgYW5kIG5vdCBndWFyZGlhbi5zdGRpbi5jbG9zZWQ6CiAgICAgICAgICAgICAgICBndWFyZGlhbi5zdGRpbi5jbG9zZSgpCiAgICAgICAgICAgIGd1YXJkaWFuLndhaXQoKQogICAgZmluYWxseToKICAgICAgICBpZiByZWFkX2ZkIGlzIG5vdCBOb25lOgogICAgICAgICAgICBvcy5jbG9zZShyZWFkX2ZkKQogICAgICAgIGlmIHdyaXRlX2ZkIGlzIG5vdCBOb25lOgogICAgICAgICAgICBvcy5jbG9zZSh3cml0ZV9mZCkKICAgICAgICBmb3Igc2lnLCBoYW5kbGVyIGluIG9sZF9oYW5kbGVycy5pdGVtcygpOgogICAgICAgICAgICBzaWduYWwuc2lnbmFsKHNpZywgaGFuZGxlcikKICAgIGlmIG5vdCByZWNlaXB0LmlzX2ZpbGUoKToKICAgICAgICBxdWFyYW50aW5lKG1hcmtlciwgImd1YXJkaWFuX21pc3NpbmdfcmVjZWlwdCIpCiAgICAgICAgcmFpc2UgUnVudGltZUVycm9yKCJHdWFyZGlhbiByZWNlaXB0IG1pc3Npbmc7IEdQVSBxdWFyYW50aW5lIHJlcXVpcmVzIG9wZXJhdG9yIHJlY29uY2lsaWF0aW9uIikKICAgIHJlc3VsdCA9IGpzb24ubG9hZHMocmVjZWlwdC5yZWFkX3RleHQoKSkKICAgIGlmIGludGVycnVwdGVkIGlzIG5vdCBOb25lOgogICAgICAgIHJlc3VsdC51cGRhdGUocGFyZW50X2V4Y2VwdGlvbj1pbnRlcnJ1cHRlZCwgaW50ZXJydXB0ZWQ9VHJ1ZSkKICAgICAgICBpZiByZXN1bHRbImV4aXRfY29kZSJdID09IDA6CiAgICAgICAgICAgIHJlc3VsdFsiZXhpdF9jb2RlIl0gPSAxMzAKICAgICAgICBxdWFyYW50aW5lKG1hcmtlciwgImRyaXZlcl9pbnRlcnJ1cHRpb246IiArIGludGVycnVwdGVkKQogICAgaWYgbm90IHJlc3VsdFsiY2xlYW51cF9jb25maXJtZWQiXToKICAgICAgICBxdWFyYW50aW5lKG1hcmtlciwgImd1YXJkaWFuX3VuY2VydGFpbl9jbGVhbnVwIikKICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoIkd1YXJkaWFuIGNsZWFudXAgdW5jZXJ0YWluOyBxdWFyYW50aW5lIHJldGFpbmVkIikKICAgIHJldHVybiByZXN1bHQKCgppZiBfX25hbWVfXyA9PSAiX19tYWluX18iOgogICAgaWYgc3lzLmFyZ3ZbMTpdICE9IFsiLS1ndWFyZGlhbiJdOgogICAgICAgIHJhaXNlIFN5c3RlbUV4aXQoIkludGVybmFsIGd1YXJkaWFuIGVudHJ5cG9pbnQgb25seSIpCiAgICByYWlzZSBTeXN0ZW1FeGl0KGd1YXJkaWFuX21haW4oKSkK', 'owned_process_tree.py': 'IyBTaGFyZWQgdGFzayBvcmNoZXN0cmF0aW9uIGZyb20gdGhlIExvUkEgbGFuZTsgbm8ga2VybmVsIG93bmVyc2hpcCBjcmVkaXQuCiIiIkxpbnV4LW9ubHkgZGVkaWNhdGVkLXJ1bm5lciBjbGVhbnVwOyBubyBHUFUgaW1wb3J0cyBvciBiYWNrZ3JvdW5kIGFjdGl2aXR5LgoKT3JpZ2luYWwgdGFzay1vcmNoZXN0cmF0aW9uIGhlbHBlci4gTm90IHBhcnQgb2Ygc29sdXRpb24vIG9yIG93bi1rZXJuZWwgY3JlZGl0LgoiIiIKaW1wb3J0IGN0eXBlcwppbXBvcnQgb3MKZnJvbSBwYXRobGliIGltcG9ydCBQYXRoCmltcG9ydCBzaWduYWwKaW1wb3J0IHN1YnByb2Nlc3MKaW1wb3J0IHN5cwppbXBvcnQgdGltZQoKCmRlZiBwcmVwYXJlKCk6CiAgICAiIiJGYWlsIGJlZm9yZSBHUFUgd29yayBpZiByYWNlLXNhZmUgcHJvY2VzcyBvd25lcnNoaXAgaXMgdW5hdmFpbGFibGUuIiIiCiAgICBpZiBub3QgaGFzYXR0cihvcywncGlkZmRfb3BlbicpIG9yIG5vdCBoYXNhdHRyKHNpZ25hbCwncGlkZmRfc2VuZF9zaWduYWwnKToKICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoJ0xpbnV4IHBpZGZkIHByb2Nlc3Mgb3duZXJzaGlwIHJlcXVpcmVkJykKICAgIGZkPW9zLnBpZGZkX29wZW4ob3MuZ2V0cGlkKCkpO29zLmNsb3NlKGZkKQogICAgbGliYz1jdHlwZXMuQ0RMTChOb25lLHVzZV9lcnJubz1UcnVlKQogICAgaWYgbGliYy5wcmN0bCgzNiwxLDAsMCwwKSE9MDoKICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoJ0Nhbm5vdCBlbmFibGUgY2hpbGQgc3VicmVhcGVyOyBubyBHUFUgam9iIHN0YXJ0ZWQnKQogICAgZW5hYmxlZD1jdHlwZXMuY19pbnQoMCkKICAgIGlmIGxpYmMucHJjdGwoMzcsY3R5cGVzLmJ5cmVmKGVuYWJsZWQpLDAsMCwwKSE9MCBvciBlbmFibGVkLnZhbHVlIT0xOgogICAgICAgIHJhaXNlIFJ1bnRpbWVFcnJvcignQ2hpbGQgc3VicmVhcGVyIHZlcmlmaWNhdGlvbiBmYWlsZWQnKQoKCmRlZiBpZGVudGl0eShwaWQpOgogICAgdHJ5OgogICAgICAgIGZpZWxkcz1QYXRoKGYnL3Byb2Mve3BpZH0vc3RhdCcpLnJlYWRfdGV4dCgpLnJzcGxpdCgnKScsMSlbMV0uc3BsaXQoKQogICAgICAgIHJldHVybiBpbnQoZmllbGRzWzFdKSxpbnQoZmllbGRzWzE5XSksZmllbGRzWzBdCiAgICBleGNlcHQgKE9TRXJyb3IsVmFsdWVFcnJvcixJbmRleEVycm9yKTpyZXR1cm4gTm9uZQoKCmRlZiBkZXNjZW5kYW50cygpOgogICAgcmVjb3Jkcz17fQogICAgZm9yIHAgaW4gUGF0aCgnL3Byb2MnKS5pdGVyZGlyKCk6CiAgICAgICAgaWYgcC5uYW1lLmlzZGlnaXQoKToKICAgICAgICAgICAgcmVjPWlkZW50aXR5KGludChwLm5hbWUpKQogICAgICAgICAgICBpZiByZWMgaXMgbm90IE5vbmU6cmVjb3Jkc1tpbnQocC5uYW1lKV09cmVjCiAgICBvd25lZD17b3MuZ2V0cGlkKCl9CiAgICB3aGlsZSBUcnVlOgogICAgICAgIGFkZD17cGlkIGZvciBwaWQsKHBhcmVudCxfLF8pIGluIHJlY29yZHMuaXRlbXMoKSBpZiBwYXJlbnQgaW4gb3duZWR9LW93bmVkCiAgICAgICAgaWYgbm90IGFkZDpicmVhawogICAgICAgIG93bmVkLnVwZGF0ZShhZGQpCiAgICByZXR1cm4ge3BpZDpyZWNvcmRzW3BpZF0gZm9yIHBpZCBpbiBvd25lZCBpZiBwaWQhPW9zLmdldHBpZCgpIGFuZCBwaWQgaW4gcmVjb3Jkc30KCgpkZWYgc2lnbmFsX2V4YWN0KHBpZCxiaXJ0aCxzaWcpOgogICAgIiIiTmV2ZXIgc2lnbmFsIGEgcmVjeWNsZWQgUElELiBUaGUgRkQgcGlucyBpZGVudGl0eSB0aHJvdWdoIHRoZSBzZW5kLiIiIgogICAgdHJ5OmZkPW9zLnBpZGZkX29wZW4ocGlkKQogICAgZXhjZXB0IFByb2Nlc3NMb29rdXBFcnJvcjpyZXR1cm4KICAgIHRyeToKICAgICAgICBub3c9aWRlbnRpdHkocGlkKQogICAgICAgIGlmIG5vdyBpcyBub3QgTm9uZSBhbmQgbm93WzFdPT1iaXJ0aDoKICAgICAgICAgICAgdHJ5OnNpZ25hbC5waWRmZF9zZW5kX3NpZ25hbChmZCxzaWcsTm9uZSwwKQogICAgICAgICAgICBleGNlcHQgUHJvY2Vzc0xvb2t1cEVycm9yOnBhc3MKICAgIGZpbmFsbHk6b3MuY2xvc2UoZmQpCgoKZGVmIF90cnlfc2lnbmFsKHBpZCxiaXJ0aCxzaWcpOgogICAgdHJ5OnNpZ25hbF9leGFjdChwaWQsYmlydGgsc2lnKQogICAgZXhjZXB0IE9TRXJyb3IgYXMgZXhjOgogICAgICAgICMgRG8gbm90IHVud2luZC9yZWxlYXNlIEdQVSBvd25lcnNoaXAgd2hlbiBjbGVhbnVwIGlzIHVuY2VydGFpbi4KICAgICAgICBxdWFyYW50aW5lKCdDbGVhbnVwIHNpZ25hbCBmYWlsZWQ6ICcrcmVwcihleGMpKQogICAgICAgIHByaW50KCdTT1VQX0NMRUFOVVBfUEVORElORyAnK3JlcHIoZXhjKSxmaWxlPXN5cy5zdGRlcnIsZmx1c2g9VHJ1ZSkKCgpkZWYgY2xlYW51cChwcm9jKToKICAgIHNhdmVkPXtzOnNpZ25hbC5zaWduYWwocyxzaWduYWwuU0lHX0lHTikgZm9yIHMgaW4gKHNpZ25hbC5TSUdJTlQsc2lnbmFsLlNJR1RFUk0pfQogICAgdHJ5OgogICAgICAgIGZyb3plbj1zZXQoKQogICAgICAgIHdoaWxlIFRydWU6CiAgICAgICAgICAgIG9ic2VydmVkPWRlc2NlbmRhbnRzKCkKICAgICAgICAgICAgbmV3PXsocGlkLHZbMV0pIGZvciBwaWQsdiBpbiBvYnNlcnZlZC5pdGVtcygpfS1mcm96ZW4KICAgICAgICAgICAgaWYgbm90IG5ldzpicmVhawogICAgICAgICAgICBmb3IgcGlkLGJpcnRoIGluIG5ldzpfdHJ5X3NpZ25hbChwaWQsYmlydGgsc2lnbmFsLlNJR1NUT1ApCiAgICAgICAgICAgIGZyb3plbi51cGRhdGUobmV3KQogICAgICAgIGZvciBwaWQsYmlydGggaW4gZnJvemVuOl90cnlfc2lnbmFsKHBpZCxiaXJ0aCxzaWduYWwuU0lHS0lMTCkKICAgICAgICBwcm9jLndhaXQoKQogICAgICAgICMgTm8gdGltZW91dC1iYXNlZCByZWxlYXNlOiBpZiBhIEdQVSBjaGlsZCBpcyBzdHVjayBpbiBrZXJuZWwgZXhpdCwKICAgICAgICAjIHJldGFpbiB0aGlzIHN1cGVydmlzb3IgYW5kIGl0cyBsb2NrIHVudGlsIGV4aXQgaXMgYWN0dWFsbHkgb2JzZXJ2ZWQuCiAgICAgICAgd2hpbGUgVHJ1ZToKICAgICAgICAgICAgd2hpbGUgVHJ1ZToKICAgICAgICAgICAgICAgIHRyeToKICAgICAgICAgICAgICAgICAgICBwaWQsXz1vcy53YWl0cGlkKC0xLG9zLldOT0hBTkcpCiAgICAgICAgICAgICAgICAgICAgaWYgbm90IHBpZDpicmVhawogICAgICAgICAgICAgICAgZXhjZXB0IENoaWxkUHJvY2Vzc0Vycm9yOmJyZWFrCiAgICAgICAgICAgIHJlbWFpbmluZz1kZXNjZW5kYW50cygpCiAgICAgICAgICAgIGlmIG5vdCByZW1haW5pbmc6YnJlYWsKICAgICAgICAgICAgZm9yIHBpZCwoXyxiaXJ0aCxfKSBpbiByZW1haW5pbmcuaXRlbXMoKTpfdHJ5X3NpZ25hbChwaWQsYmlydGgsc2lnbmFsLlNJR0tJTEwpCiAgICAgICAgICAgIHRpbWUuc2xlZXAoLjA1KQogICAgZmluYWxseToKICAgICAgICBmb3Igc2lnLGhhbmRsZXIgaW4gc2F2ZWQuaXRlbXMoKTpzaWduYWwuc2lnbmFsKHNpZyxoYW5kbGVyKQoKCmRlZiBydW4oYXJndiwqLGN3ZCxlbnYsc3Rkb3V0LHRpbWVvdXQ9Tm9uZSxzdGRlcnI9c3VicHJvY2Vzcy5TVERPVVQpOgogICAgIiIiUnVuIGluIGRlZGljYXRlZCBzdXBlcnZpc29yIG9ubHk7IGNsZWFudXAgbmVzdGVkIHNlc3Npb25zIG9uIGFueSBmYWlsdXJlLiIiIgogICAgcHJvYz1zdWJwcm9jZXNzLlBvcGVuKGFyZ3YsY3dkPWN3ZCxlbnY9ZW52LHN0ZG91dD1zdGRvdXQsc3RkZXJyPXN0ZGVycixzdGFydF9uZXdfc2Vzc2lvbj1UcnVlKQogICAgdHJ5OgogICAgICAgIGNvZGU9cHJvYy53YWl0KHRpbWVvdXQ9dGltZW91dCkKICAgICAgICBpZiBkZXNjZW5kYW50cygpOgogICAgICAgICAgICBxdWFyYW50aW5lKCdDaGlsZCBleGl0ZWQgbGVhdmluZyBkZXNjZW5kYW50cycpCiAgICAgICAgICAgIGNsZWFudXAocHJvYykKICAgICAgICAgICAgcmFpc2UgUnVudGltZUVycm9yKCdDaGlsZCBleGl0ZWQgbGVhdmluZyBkZXNjZW5kYW50czsgY2xlYW5lZCBhbmQgcmVqZWN0ZWQnKQogICAgICAgIHJldHVybiBjb2RlCiAgICBleGNlcHQgQmFzZUV4Y2VwdGlvbiBhcyBleGM6CiAgICAgICAgcXVhcmFudGluZSgnT3V0ZXIgdGltZW91dC9pbnRlcnJ1cHRpb24vZXJyb3I6ICcrcmVwcihleGMpKQogICAgICAgIGNsZWFudXAocHJvYykKICAgICAgICByYWlzZQoKCmRlZiBpbnN0YWxsX3NpZ25hbF9oYW5kbGVycygpOgogICAgZGVmIGludGVycnVwdGVkKHNpZ251bSxmcmFtZSk6CiAgICAgICAgcXVhcmFudGluZSgnUnVubmVyIGludGVycnVwdGVkIGJ5IHNpZ25hbCAnK3N0cihzaWdudW0pKQogICAgICAgIHJhaXNlIEludGVycnVwdGVkRXJyb3IoJ1J1bm5lciBpbnRlcnJ1cHRlZCBieSBzaWduYWwgJytzdHIoc2lnbnVtKSkKICAgIGZvciBzaWcgaW4gKHNpZ25hbC5TSUdJTlQsc2lnbmFsLlNJR1RFUk0pOnNpZ25hbC5zaWduYWwoc2lnLGludGVycnVwdGVkKQoKIyBBIGNyYXNoLXN1cnZpdmluZyBtYXJrZXIgaXMgcGxhY2VkIGJlZm9yZSBhbnkgR1BVIHN1YnByb2Nlc3MuIE9ubHkgaXRzIGV4YWN0CiMgb3duZXIgbWF5IGNsZWFyIGl0IG9uIGEgY2xlYW4sIG5vbi1pbnRlcnJ1cHRlZCBleGl0IHdpdGggbm8gbGl2ZSBkZXNjZW5kYW50cy4KIyBJbnRlcnJ1cHRlZC90aW1lZC1vdXQgd29yayBsZWF2ZXMgaXQgZm9yIGV4cGxpY2l0IG9wZXJhdG9yIHJlY29uY2lsaWF0aW9uLgpfTUFSS0VSID0gTm9uZQpfVE9LRU4gPSBOb25lCl9LRUVQX01BUktFUiA9IEZhbHNlCgoKZGVmIGNoZWNrX3F1YXJhbnRpbmUocGF0aCk6CiAgICBpZiBQYXRoKHBhdGgpLmV4aXN0cygpOgogICAgICAgIHJhaXNlIFJ1bnRpbWVFcnJvcignR1BVMCBxdWFyYW50aW5lZDogcmVjb25jaWxlICcrc3RyKHBhdGgpKycgYmVmb3JlIGFub3RoZXIgcnVuJykKCgpkZWYgYmVnaW5fcXVhcmFudGluZShwYXRoKToKICAgIGdsb2JhbCBfTUFSS0VSLF9UT0tFTixfS0VFUF9NQVJLRVIKICAgIGltcG9ydCBqc29uLHV1aWQKICAgIF9NQVJLRVI9UGF0aChwYXRoKTtfVE9LRU49dXVpZC51dWlkNCgpLmhleDtfS0VFUF9NQVJLRVI9RmFsc2UKICAgIGZkPW9zLm9wZW4oc3RyKF9NQVJLRVIpLG9zLk9fV1JPTkxZfG9zLk9fQ1JFQVR8b3MuT19FWENMfG9zLk9fTk9GT0xMT1csMG82MDApCiAgICB3aXRoIG9zLmZkb3BlbihmZCwndycpIGFzIG91dDoKICAgICAgICBqc29uLmR1bXAoeydvd25lcl90b2tlbic6X1RPS0VOLCdvd25lcl9waWQnOm9zLmdldHBpZCgpLCdzdGF0dXMnOidPV05FRF9HUFVfV09SS19JTl9QUk9HUkVTUyd9LG91dCkKICAgICAgICBvdXQuZmx1c2goKTtvcy5mc3luYyhvdXQuZmlsZW5vKCkpCgoKZGVmIHF1YXJhbnRpbmUocmVhc29uKToKICAgIGdsb2JhbCBfS0VFUF9NQVJLRVIKICAgIF9LRUVQX01BUktFUj1UcnVlCiAgICBpZiBfTUFSS0VSIGlzIE5vbmU6cmV0dXJuCiAgICBpbXBvcnQganNvbgogICAgdHJ5OnZhbHVlPWpzb24ubG9hZHMoX01BUktFUi5yZWFkX3RleHQoKSkKICAgIGV4Y2VwdCAoT1NFcnJvcixWYWx1ZUVycm9yKTpyZXR1cm4KICAgIGlmIHZhbHVlLmdldCgnb3duZXJfdG9rZW4nKSE9X1RPS0VOOnJldHVybgogICAgdmFsdWUudXBkYXRlKHN0YXR1cz0nQkxPQ0tFRF9PUEVSQVRPUl9SRUNPTkNJTElBVElPTl9SRVFVSVJFRCcscmVhc29uPXN0cihyZWFzb24pKQogICAgX01BUktFUi53cml0ZV90ZXh0KGpzb24uZHVtcHModmFsdWUpKydcbicpCgoKZGVmIGZpbmlzaF9xdWFyYW50aW5lKCk6CiAgICBpZiBfTUFSS0VSIGlzIE5vbmUgb3IgX0tFRVBfTUFSS0VSOnJldHVybgogICAgaWYgZGVzY2VuZGFudHMoKToKICAgICAgICBxdWFyYW50aW5lKCdMaXZlIG93bmVkIGRlc2NlbmRhbnRzIHJlbWFpbiBhdCBzdXBlcnZpc29yIGV4aXQnKQogICAgICAgIHJldHVybgogICAgaW1wb3J0IGpzb24KICAgIHRyeTp2YWx1ZT1qc29uLmxvYWRzKF9NQVJLRVIucmVhZF90ZXh0KCkpCiAgICBleGNlcHQgKE9TRXJyb3IsVmFsdWVFcnJvcik6cmV0dXJuCiAgICBpZiB2YWx1ZS5nZXQoJ293bmVyX3Rva2VuJyk9PV9UT0tFTjpfTUFSS0VSLnVubGluaygpCg==', 'runtime_probe.py': 'IiIiUmVhZC1vbmx5LCBzaG9ydC1saXZlZCBkZXBsb3llZC10ZW1wbGF0ZSBnYXRlLiBOZXZlciBwcmludHMgc2VjcmV0IHZhbHVlcy4iIiIKaW1wb3J0IGltcG9ydGxpYi5tZXRhZGF0YSBhcyBtZXRhZGF0YSxqc29uLG9zLHBsYXRmb3JtLHN1YnByb2Nlc3Msc3lzCmZyb20gcGF0aGxpYiBpbXBvcnQgUGF0aApFWFBFQ1RFRD17J3RvcmNoJzonMi4xMS4wK2N1MTI4JywndHJpdG9uJzonMy42LjAnLCdudW1weSc6JzIuMy41Jywnc291cC1jbGknOicwLjc3LjAnLCd0cmFuc2Zvcm1lcnMnOic1LjE2LjEnLCdwZWZ0JzonMC4yMC4wJywndHJsJzonMC4yOS4xJywnYml0c2FuZGJ5dGVzJzonMC41MC4yJywnYWNjZWxlcmF0ZSc6JzEuMTQuMCcsJ2RhdGFzZXRzJzonNC44LjUnfQoKZGVmIG1haW4oKToKIG91dD1QYXRoKHN5cy5hcmd2WzFdKTtyPXsncGFzc2VkJzpGYWxzZSwnZ3B1X3dvcmtsb2FkX2V4ZWN1dGVkJzpGYWxzZSwnc2NvcmUnOk5vbmUsJ2FkbWlzc2lvbic6J1VOS05PV04nfQogdHJ5OgogIHZlcnNpb25zPXtrOm1ldGFkYXRhLnZlcnNpb24oaykgZm9yIGsgaW4gRVhQRUNURUR9CiAgci51cGRhdGUocHl0aG9uPXBsYXRmb3JtLnB5dGhvbl92ZXJzaW9uKCkscGFja2FnZXM9dmVyc2lvbnMsdWlkPW9zLmdldHVpZCgpLGNwdV9hZmZpbml0eT1zb3J0ZWQob3Muc2NoZWRfZ2V0YWZmaW5pdHkoMCkpKQogIGFzc2VydCBwbGF0Zm9ybS5weXRob25fdmVyc2lvbigpPT0nMy4xMy4xNScsJ1B5dGhvbiBwaW4gbWlzbWF0Y2gnCiAgYXNzZXJ0IHZlcnNpb25zPT1FWFBFQ1RFRCwnRXhhY3QgcGFja2FnZSBwaW5zIG1pc21hdGNoJwogIGFzc2VydCBvcy5nZXR1aWQoKT09MTAwMDEsJ0RlcGxveWVkIHJ1bnRpbWUgdWlkIG1pc21hdGNoJwogIGFzc2VydCBsZW4ob3Muc2NoZWRfZ2V0YWZmaW5pdHkoMCkpPT00LCdGb3VyIGFzc2lnbmVkIENQVXMgcmVxdWlyZWQnCiAgZm9yIHAsd2FudGVkIGluIFsoJ21lbW9yeS5tYXgnLHN0cigzMCoxMDI0KiozKSksKCdtZW1vcnkuc3dhcC5tYXgnLCcwJyksKCdjcHVzZXQuY3B1cy5lZmZlY3RpdmUnLCcwLTMnKV06CiAgIGFjdHVhbD0oUGF0aCgnL3N5cy9mcy9jZ3JvdXAnKS9wKS5yZWFkX3RleHQoKS5zdHJpcCgpO3JbcF09YWN0dWFsCiAgIGFzc2VydCBhY3R1YWw9PXdhbnRlZCwnRGVwbG95ZWQgY2dyb3VwIG1pc21hdGNoOiAnK3AKICBhc3NlcnQgbm90IFBhdGgoJy92YXIvcnVuL2RvY2tlci5zb2NrJykuZXhpc3RzKCksJ1VuZXhwZWN0ZWQgRG9ja2VyIHNvY2tldCcKICBhc3NlcnQgbm90IFBhdGgoJy9ldGMvc291cC1xdWV1ZS93b3JrZXIuZW52JykuZXhpc3RzKCksJ1VuZXhwZWN0ZWQgd29ya2VyIGVudmlyb25tZW50IGZpbGUnCiAgYXNzZXJ0IG5vdCBhbnkoJ1RPS0VOJyBpbiBrIGFuZCBrIT0nVE9LRU5JWkVSU19QQVJBTExFTElTTScgZm9yIGsgaW4gb3MuZW52aXJvbiksJ1Rva2VuLWJlYXJpbmcgZW52aXJvbm1lbnQgcmVqZWN0ZWQgd2l0aG91dCByZWFkaW5nIGl0cyB2YWx1ZScKICBhc3NlcnQgbm90IG9zLmVudmlyb24uZ2V0KCdUUklUT05fSU5URVJQUkVUJyksJ0ludGVycHJldGVyIG1vZGUgZm9yYmlkZGVuJwogIGltcG9ydCB0b3JjaCx0cml0b24sbnVtcHkKICBhc3NlcnQgdG9yY2guX192ZXJzaW9uX189PScyLjExLjArY3UxMjgnIGFuZCB0b3JjaC52ZXJzaW9uLmN1ZGE9PScxMi44JwogIGFzc2VydCB0cml0b24uX192ZXJzaW9uX189PSczLjYuMCcgYW5kIG51bXB5Ll9fdmVyc2lvbl9fPT0nMi4zLjUnCiAgYXNzZXJ0IHRvcmNoLmN1ZGEuZGV2aWNlX2NvdW50KCk9PTEgYW5kIHRvcmNoLmN1ZGEuZ2V0X2RldmljZV9uYW1lKDApPT0nVGVzbGEgVDQnCiAgYXNzZXJ0IHRvcmNoLmN1ZGEuZ2V0X2RldmljZV9jYXBhYmlsaXR5KDApPT0oNyw1KQogIHIudXBkYXRlKGdwdV9uYW1lPXRvcmNoLmN1ZGEuZ2V0X2RldmljZV9uYW1lKDApLGN1ZGFfYnVpbGQ9dG9yY2gudmVyc2lvbi5jdWRhLHRvcmNoX2dpdD10b3JjaC52ZXJzaW9uLmdpdF92ZXJzaW9uKQogIGZyb20gaHVnZ2luZ2ZhY2VfaHViLmNvbnN0YW50cyBpbXBvcnQgSEZfSFVCX0NBQ0hFCiAgclsncmVzb2x2ZWRfcHVibGljX2h1Yl9jYWNoZSddPXN0cihQYXRoKEhGX0hVQl9DQUNIRSkucmVzb2x2ZSgpKQogIHJbJ3Bhc3NlZCddPVRydWUKIGV4Y2VwdCBCYXNlRXhjZXB0aW9uIGFzIGU6clsnZXJyb3InXT1yZXByKGUpCiBvdXQud3JpdGVfdGV4dChqc29uLmR1bXBzKHIsaW5kZW50PTIpKydcbicpO3ByaW50KGpzb24uZHVtcHMocixpbmRlbnQ9MikpCiByYWlzZSBTeXN0ZW1FeGl0KDAgaWYgclsncGFzc2VkJ10gZWxzZSAxKQppZiBfX25hbWVfXz09J19fbWFpbl9fJzptYWluKCkK', 'official_manifest.json': 'ewogICJoYXJuZXNzL190cmFpbi5weSI6ICI0NWE2MjY0MmRjZWFiMzJjZjNlZmNkYTUwNGRlODdhOWMyNWVhMjhiMTE5YjE4MDZmNTZkMTRmZjExNDZhODlkIiwKICAiaGFybmVzcy9jaGVja19ydWxlcy5weSI6ICI4NzlhYjE4NjE2NzRmNmU3MTRlYWRkMDBhYWYwNDQyMzVkOTg1MjZkZDc1ZmFiMTAzNWRlZWMzNDAxMTIxZWY1IiwKICAiaGFybmVzcy9jb21tb24ucHkiOiAiYzgzMDY2ZjI0ZjI2NmRhYjYyZWJjYzljMmQ0NGY1YTY1OTAyNzYwYjUzZGE5MTJlNjVlYmZlNTExNTc5YzJkYiIsCiAgImhhcm5lc3MvZXZhbF9hZGFwdGVyLnB5IjogIjQ3Y2RkYmI4ZTQ1NzljNjlmOTUzNWIwOTlmZjIzNzFiMWMzYWYwOGJmYmQ5M2QxNGJmNzlhYWRkNzRkMzg0ODIiLAogICJoYXJuZXNzL2V4YW1wbGVfcmVmZXJlbmNlLnB5IjogIjZhMTEwNTVmNGExODIyZTUzOTcxYzIyOGM2ZTNlM2FhNmI4OGVkZDllYjZkYjgxYTM4OGUyNmE3MGEwMzBjZWIiLAogICJoYXJuZXNzL3J1bl90YXNrLnB5IjogImQ5MzVhODk1OWYzNDg5NzczMTNmMmVlZjM2ODcwZmYyNjk5YjliYTAxM2E3OTc4OTE4ZmYzNWUxNDg5NjhhNzkiLAogICJoYXJuZXNzL3Njb3JlLnB5IjogIjczMWM5YjY0NGRjYzQwNjliMGIwNjk2MTZmNDM4MDY0ZDFmN2YzNzZhMjAyZmViMTUyMDFmZDkxMGVjNjg4YzUiLAogICJ0YXNrL3ByZXBhcmVfZGF0YS5weSI6ICJhODU0M2YxMWE1NWFmZjgwZGJkYzcxNGEwZTYyMzkxMjQ0YTUyOTY2MTZlZWEyZTM0ZGMyZWIzZmYzZWMxOGY3IiwKICAidGFzay90YXNrLnlhbWwiOiAiZGQ3MGM3MzlmY2QxNDNlMDZhZWYzYmIxZWZjYzkzODZmNjgyYmYxZGUyNmYzMGUwODQwOWQ4NmE5N2ZmNDdjZSIsCiAgInRlc3RzL2NvbmZ0ZXN0LnB5IjogImE4MmVhYmFiNDY4ZWEwZWYxZTAwM2NjY2Q3YzM4MjdjZTJmNGQwMGQwOWMzMjk0OGEzNGZiNDgyMjFmNzMzN2EiLAogICJ0ZXN0cy90ZXN0X2NhbGxiYWNrLnB5IjogIjE5MTkzMDNhMzY3MGMwYTMzYWJiZmNlOWIzM2VhYjA5MTNiYjY0YTNhMjFjMzMxMmQ4NjUzMWMzMmI1YmY0OTAiLAogICJ0ZXN0cy90ZXN0X2NvbmZpZy5weSI6ICJkYjgwOGM0NDQ3MGQxNDRiN2NmMThiNzMyOTQ5MmY0MWY3NmUzN2NmYzI0MzA2ZTdjYmEwNDg4YmNhOWQ3ODdjIiwKICAidGVzdHMvdGVzdF9mb3JtYXRzLnB5IjogIjVkNjU2ZWQ1MGM1OWY3OTRlOTM0NDczZDQ1OWVhODU2MzVmODkyYmEwMjhlMTkyNDE0NDc0ZTliNWQ2NTkxOWUiLAogICJ0ZXN0cy90ZXN0X2xvYWRlci5weSI6ICIzZGFkMWJlYTJkMzNhYWI5N2U1MmI3Y2Q0NDg4OGQ3NjhiYzk1MTlhZjQ0ZmJlNmNmM2U5NmMyNjAwMjY2MzJmIiwKICAidGVzdHMvdGVzdF90cmFpbmVyX2luaXQucHkiOiAiYjc2MDhkNjJhNDA5ODNiZjdjOWU1YjI3MGUwZTUyN2I2NDFhODM3ZTQ2MmQ2ZGYyOTA5ZmVjODhmNjA3ZTgyMSIKfQo=', 'reviewed_support.py': 'aW1wb3J0IGFyZ3BhcnNlLGZjbnRsLGhhc2hsaWIsanNvbixtYXRoLG9zLHNodXRpbCxzeXMsdGhyZWFkaW5nLHRpbWUsdXVpZCx6aXBmaWxlCgpmcm9tIHBhdGhsaWIgaW1wb3J0IFBhdGgKCmltcG9ydCBwcm9jZXNzX2d1YXJkCgppbXBvcnQgb3duZWRfcHJvY2Vzc190cmVlIGFzIHRyZWUKCmRlZiBzaGEocCk6cmV0dXJuIGhhc2hsaWIuc2hhMjU2KFBhdGgocCkucmVhZF9ieXRlcygpKS5oZXhkaWdlc3QoKQoKZGVmIHJlYWQocCk6cmV0dXJuIGpzb24ubG9hZHMoUGF0aChwKS5yZWFkX3RleHQoKSkKCmRlZiB3cml0ZShwLHZhbHVlKToKIHA9UGF0aChwKTtwZW5kaW5nPXAud2l0aF9uYW1lKHAubmFtZSsnLnBlbmRpbmcnKTtwZW5kaW5nLndyaXRlX3RleHQoanNvbi5kdW1wcyh2YWx1ZSxpbmRlbnQ9MixhbGxvd19uYW49RmFsc2UpKydcbicpO3BlbmRpbmcucmVwbGFjZShwKQoKZGVmIHZlcmlmeV9vZmZpY2lhbChyZXBvLG1hbmlmZXN0KToKIGlmIG5vdCByZXBvLmlzX2RpcigpIG9yIHJlcG8uaXNfc3ltbGluaygpOnJhaXNlIFJ1bnRpbWVFcnJvcignU09VUF9SRVBPIG11c3QgYmUgYSByZWFsIHByZXBhcmVkIGpvYiBkaXJlY3RvcnknKQogZm9yIG5hbWUsd2FudGVkIGluIG1hbmlmZXN0Lml0ZW1zKCk6CiAgcD1yZXBvL25hbWUKICBpZiBub3QgcC5pc19maWxlKCkgb3IgcC5pc19zeW1saW5rKCkgb3Igc2hhKHApIT13YW50ZWQ6cmFpc2UgUnVudGltZUVycm9yKCdQcm90ZWN0ZWQgb2ZmaWNpYWwgc291cmNlIG1pc21hdGNoOiAnK25hbWUpCiBhY3R1YWw9e3N0cihwLnJlbGF0aXZlX3RvKHJlcG8pKSBmb3IgZm9sZGVyIGluICgnaGFybmVzcycsJ3Rhc2snLCd0ZXN0cycpIGZvciBwIGluIChyZXBvL2ZvbGRlcikucmdsb2IoJyonKQogICAgICAgICBpZiBwLmlzX2ZpbGUoKSBhbmQgJ19fcHljYWNoZV9fJyBub3QgaW4gcC5wYXJ0cyBhbmQgcC5zdWZmaXggbm90IGluICgnLnB5YycsJy5weW8nKX0KIGlmIGFjdHVhbCE9c2V0KG1hbmlmZXN0KTpyYWlzZSBSdW50aW1lRXJyb3IoJ1VuZXhwZWN0ZWQgcHJvdGVjdGVkIG9mZmljaWFsIGZpbGVzJykKCkVYUE9SVF9CVURHRVQ9NTYqMTAyNCoqMgoKRVhQT1JUX09NSVRURUQ9e30KCmRlZiBleHBvcnRfc21hbGwocnVuLGV4cG9ydCk6CiAiIiJSZWFkLW9ubHkgaW5jcmVtZW50YWwgZXhwb3J0OyBubyBhZGFwdGVycywgd2VpZ2h0cyBvciBjb21waWxlciBjYWNoZXMuIiIiCiB1c2VkPXN1bShwLnN0YXQoKS5zdF9zaXplIGZvciBwIGluIGV4cG9ydC5yZ2xvYignKicpIGlmIHAuaXNfZmlsZSgpKQogZXhjbHVkZWQ9eydhZGFwdGVyJywndHJpdG9uX2NhY2hlJywnaW5kdWN0b3JfY2FjaGUnLCdwcmVmbGlnaHRfdHJpdG9uJywncHJlZmxpZ2h0X2luZHVjdG9yJywnb3JpZ2luYWxfc29sdXRpb24nfQogZm9yIGJhc2UsZGlycyxmaWxlcyBpbiBvcy53YWxrKHJ1bik6CiAgZGlyc1s6XT1bZCBmb3IgZCBpbiBkaXJzIGlmIGQgbm90IGluIGV4Y2x1ZGVkIGFuZCBub3QgZC5zdGFydHN3aXRoKCdjaGVja3BvaW50LScpIGFuZCBub3QgKFBhdGgoYmFzZSkvZCkuaXNfc3ltbGluaygpXQogIGZvciBuYW1lIGluIGZpbGVzOgogICBwYXRoPVBhdGgoYmFzZSkvbmFtZTtyZWw9cGF0aC5yZWxhdGl2ZV90byhydW4pCiAgIHRyeToKICAgIGlmIG5vdCBwYXRoLmlzX3N5bWxpbmsoKSBhbmQgcGF0aC5zdWZmaXggaW4gKCcuanNvbicsJy5qc29ubCcsJy5sb2cnLCcueWFtbCcpOgogICAgIGlmIHBhdGguc3RhdCgpLnN0X3NpemU+PTQqMTAyNCoqMjoKICAgICAgRVhQT1JUX09NSVRURURbc3RyKHJlbCldPSdwZXJfZmlsZV9saW1pdF80TWlCJztjb250aW51ZQogICAgIGRlc3Q9ZXhwb3J0LydyYXcnL3JlbDtkZXN0LnBhcmVudC5ta2RpcihwYXJlbnRzPVRydWUsZXhpc3Rfb2s9VHJ1ZSkKICAgICBwcmlvcj1kZXN0LnN0YXQoKS5zdF9zaXplIGlmIGRlc3QuZXhpc3RzKCkgZWxzZSAwCiAgICAgc2l6ZT1wYXRoLnN0YXQoKS5zdF9zaXplCiAgICAgaWYgdXNlZC1wcmlvcitzaXplPkVYUE9SVF9CVURHRVQ6CiAgICAgIEVYUE9SVF9PTUlUVEVEW3N0cihyZWwpXT0nYWdncmVnYXRlX2J1ZGdldF81Nk1pQic7Y29udGludWUKICAgICB0ZW1wPWRlc3Qud2l0aF9uYW1lKGRlc3QubmFtZSsnLmNvcHlpbmcnKTtzaHV0aWwuY29weWZpbGUocGF0aCx0ZW1wKTt0ZW1wLnJlcGxhY2UoZGVzdCk7dXNlZD11c2VkLXByaW9yK3NpemUKICAgZXhjZXB0IChGaWxlTm90Rm91bmRFcnJvcixPU0Vycm9yKTpwYXNzCgpjbGFzcyBFeHBvcnRlcjoKIGRlZiBfX2luaXRfXyhzZWxmLHJ1bixleHBvcnQpOnNlbGYucnVuPXJ1bjtzZWxmLmV4cG9ydD1leHBvcnQ7c2VsZi5zdG9wPXRocmVhZGluZy5FdmVudCgpO3NlbGYudGhyZWFkPU5vbmUKIGRlZiBfX2VudGVyX18oc2VsZik6CiAgZGVmIGxvb3AoKToKICAgd2hpbGUgbm90IHNlbGYuc3RvcC53YWl0KDEpOmV4cG9ydF9zbWFsbChzZWxmLnJ1bixzZWxmLmV4cG9ydCkKICBzZWxmLnRocmVhZD10aHJlYWRpbmcuVGhyZWFkKHRhcmdldD1sb29wLGRhZW1vbj1UcnVlKTtzZWxmLnRocmVhZC5zdGFydCgpO3JldHVybiBzZWxmCiBkZWYgX19leGl0X18oc2VsZiwqZXhjKTpzZWxmLnN0b3Auc2V0KCk7c2VsZi50aHJlYWQuam9pbigpO2V4cG9ydF9zbWFsbChzZWxmLnJ1bixzZWxmLmV4cG9ydCkKCmRlZiBpZGxlX2NoZWNrKGVudixyZXBvLHJ1bixsb2NrZmQpOgogY29kZT0nJydpbXBvcnQganNvbixzdWJwcm9jZXNzLHN5cwpwPXN1YnByb2Nlc3MucnVuKFsnbnZpZGlhLXNtaScsJy0tcXVlcnktY29tcHV0ZS1hcHBzPXBpZCx1c2VkX2dwdV9tZW1vcnknLCctLWZvcm1hdD1jc3Ysbm9oZWFkZXIsbm91bml0cyddLGNhcHR1cmVfb3V0cHV0PVRydWUsdGV4dD1UcnVlLGNoZWNrPVRydWUpCmFzc2VydCBub3QgcC5zdGRvdXQuc3RyaXAoKSwnR1BVIHByb2Nlc3NlcyBzdGlsbCBhY3RpdmUgYWZ0ZXIgcHJvYmUvcHJlZmxpZ2h0JwpyPXN1YnByb2Nlc3MucnVuKFsnbnZpZGlhLXNtaScsJy0tcXVlcnktZ3B1PWluZGV4LG1lbW9yeS51c2VkJywnLS1mb3JtYXQ9Y3N2LG5vaGVhZGVyLG5vdW5pdHMnXSxjYXB0dXJlX291dHB1dD1UcnVlLHRleHQ9VHJ1ZSxjaGVjaz1UcnVlKQpyb3dzPVtsaW5lLnNwbGl0KCcsJykgZm9yIGxpbmUgaW4gci5zdGRvdXQuc3BsaXRsaW5lcygpIGlmIGxpbmUuc3RyaXAoKV0KYXNzZXJ0IHJvd3MgYW5kIHN1bShpbnQoeFsxXSkgZm9yIHggaW4gcm93cyk8MTAwMCwnR1BVIHBoeXNpY2FsIGludmVudG9yeSBub3QgaWRsZScKcHJpbnQoanNvbi5kdW1wcyh7J2lkbGUnOlRydWUsJ3BoeXNpY2FsX21lbW9yeV9yb3dzJzpyb3dzfSkpJycnCiByZXR1cm4gcHJvY2Vzc19ndWFyZC5ydW4oW3N5cy5leGVjdXRhYmxlLCctYycsY29kZV0sY3dkPXJlcG8sZW52PWVudixsb2c9cnVuLydpZGxlLmxvZycsdGltZW91dD0yMCxsb2NrX2ZkPWxvY2tmZCkK', 'worker.py': 'IiIiQm91bmRlZCBjb21wb25lbnQgdGltaW5nLCBuZXZlciBmdWxsIHRyYWluaW5nIG9yIG1vZGVsLW1lbW9yeSBtZWFzdXJlbWVudC4iIiIKaW1wb3J0IGFyZ3BhcnNlLGhhc2hsaWIsaW5zcGVjdCxqc29uLG9zLHN0YXRpc3RpY3MsdGltZSx0cmFjZWJhY2sKZnJvbSBwYXRobGliIGltcG9ydCBQYXRoClRTPSgxMzMsMjM1LDMxNyw0ODUsNTEyKTtHUk9VUFM9NTtSRVBFVElUSU9OUz04O1dBUk1VUD0zClZBUklBTlRTPXsnbmF0aXZlJzonZXhwYW5kZWRfcXdlbl9hZGRfbm9ybScsJ3N0YW5kYWxvbmUnOidoaXN0b3JpY2FsX29yaWdpbmFsX3RyaWFsMDAxX25vcm1fYWZ0ZXJfbmF0aXZlX2FkZCcsJ2Z1c2VkJzonb3JpZ2luYWxfdHdvX291dHB1dF9yZXNpZHVhbF9ub3JtJ30KZGVmIHdyaXRlKHBhdGgsdmFsdWUpOnBhdGgud3JpdGVfdGV4dChqc29uLmR1bXBzKHZhbHVlLGluZGVudD0yLGFsbG93X25hbj1GYWxzZSkrJ1xuJykKZGVmIG1haW4oKToKIHA9YXJncGFyc2UuQXJndW1lbnRQYXJzZXIoKTtwLmFkZF9hcmd1bWVudCgnLS1tZXRob2QnLGNob2ljZXM9dHVwbGUoVkFSSUFOVFMpLHJlcXVpcmVkPVRydWUpO3AuYWRkX2FyZ3VtZW50KCctLW91dCcscmVxdWlyZWQ9VHJ1ZSk7YT1wLnBhcnNlX2FyZ3MoKTtvdXQ9UGF0aChhLm91dCkKIHJlcG9ydD17J3Njb3BlJzonTUlDUk9CRU5DSF9PTkxZJywnc3RhdHVzJzonRkFJTEVEX01JQ1JPQkVOQ0hfT05MWScsJ21ldGhvZCc6YS5tZXRob2QsJ21ldGhvZF92YXJpYW50cyc6VkFSSUFOVFMsJ3RyYWluaW5nX2V4ZWN1dGVkJzpGYWxzZSwnY29tcGxldGVfZml4ZWRfdGFzayc6RmFsc2UsJ3Njb3JlJzpOb25lLCdhZG1pc3Npb24nOidVTktOT1dOJywncm93cyc6W10sCiAnbWVhc3VyZW1lbnRfY29udHJhY3QnOnsnY29sZCc6J1N5bmNocm9uaXplZCBmaXJzdCBjb21wb25lbnQgY2FsbCBpbiBwcml2YXRlIGluaXRpYWxseSBlbXB0eSBjb21waWxlciBjYWNoZTsgaW5jbHVkZXMgYWxsb2NhdGlvbi9kaXNwYXRjaC9jb21waWxlL2RldmljZSB3b3JrLCBub3QgcHVyZSBjb21waWxlIHRpbWUuJywnd2FybSc6J0NVREEtZXZlbnQgc3RyZWFtIGludGVydmFscyBpbmNsdWRlIFB5dGhvbi9hdXRvZ3JhZCBkaXNwYXRjaCBnYXBzOyBub3Qga2VybmVsIHN1bXMuJywnb3JkZXInOidGaXhlZCBuYXRpdmUvc3RhbmRhbG9uZS9mdXNlZCBvcmRlciwgbm8gY291bnRlcmJhbGFuY2VkIHBlcmZvcm1hbmNlIGNsYWltLicsJ2NhY2hlJzonRnJlc2ggY2hpbGQgcGVyIG1ldGhvZDsgY2FjaGUgcmV1c2VkIGFjcm9zcyBzaGFwZXMgb25seSBpbnNpZGUgY2hpbGQuJywnYm90aF9vdXRwdXRzJzonRlAzMiBub3JtIG91dHB1dCBhbmQgcm91bmRlZCBGUDMyIHJlc2lkdWFsOyBib3RoIGNvdGFuZ2VudHMgcGFydGljaXBhdGUuJywnZ3JvdXBzJzpHUk9VUFMsJ3JlcGV0aXRpb25zJzpSRVBFVElUSU9OU319CiB3cml0ZShvdXQscmVwb3J0KQogdHJ5OgogIGFzc2VydCBub3Qgb3MuZW52aXJvbi5nZXQoJ1RSSVRPTl9JTlRFUlBSRVQnKTtjYWNoZT1QYXRoKG9zLmVudmlyb25bJ1RSSVRPTl9DQUNIRV9ESVInXSk7YXNzZXJ0IGNhY2hlLmlzX2RpcigpIGFuZCBub3QgYW55KGNhY2hlLml0ZXJkaXIoKSkKICBpbXBvcnQgdG9yY2gsdHJpdG9uLHRyYW5zZm9ybWVycwogIGZyb20gdHJhbnNmb3JtZXJzLm1vZGVscy5xd2VuMiBpbXBvcnQgbW9kZWxpbmdfcXdlbjIgYXMgcXdlbgogIGZyb20gc29sdXRpb24ucmVzaWR1YWxfbm9ybSBpbXBvcnQgZnVzZWRfcmVzaWR1YWxfbm9ybQogIGZyb20gc29sdXRpb24uY29udHJhY3RzIGltcG9ydCBmaXh0dXJlLG5hdGl2ZSxlbmdpbmVlcmluZ19jbG9zZQogIGZyb20gc29sdXRpb24ud2ViX3ByZWZsaWdodCBpbXBvcnQgRVhQRUNURURfUVdFTl9CTE9CCiAgZnJvbSBvbGRfc29sdXRpb24ucm1zbm9ybSBpbXBvcnQgZnJvemVuX3JtczMyCiAgYXNzZXJ0IHRvcmNoLl9fdmVyc2lvbl9fPT0nMi4xMS4wK2N1MTI4JyBhbmQgdHJpdG9uLl9fdmVyc2lvbl9fPT0nMy42LjAnIGFuZCB0cmFuc2Zvcm1lcnMuX192ZXJzaW9uX189PSc1LjE2LjEnCiAgYXNzZXJ0IHRvcmNoLmN1ZGEuZGV2aWNlX2NvdW50KCk9PTEgYW5kIHRvcmNoLmN1ZGEuZ2V0X2RldmljZV9uYW1lKDApPT0nVGVzbGEgVDQnIGFuZCB0b3JjaC5jdWRhLmdldF9kZXZpY2VfY2FwYWJpbGl0eSgwKT09KDcsNSkKICByYXc9UGF0aChpbnNwZWN0LmdldGZpbGUocXdlbikpLnJlYWRfYnl0ZXMoKTthc3NlcnQgaGFzaGxpYi5zaGExKGInYmxvYiAnK3N0cihsZW4ocmF3KSkuZW5jb2RlKCkrYidcMCcrcmF3KS5oZXhkaWdlc3QoKT09RVhQRUNURURfUVdFTl9CTE9CCiAgZGVmIHN0YW5kYWxvbmUoeCxiLHcpOno9eCtiO3JldHVybiBmcm96ZW5fcm1zMzIoeix3KSx6CiAgZm49eyduYXRpdmUnOm5hdGl2ZSwnc3RhbmRhbG9uZSc6c3RhbmRhbG9uZSwnZnVzZWQnOmZ1c2VkX3Jlc2lkdWFsX25vcm19W2EubWV0aG9kXQogIGRlZiB3YWxsKGNhbGwpOgogICB0b3JjaC5jdWRhLnN5bmNocm9uaXplKCk7c3RhcnQ9dGltZS5wZXJmX2NvdW50ZXIoKTt2YWx1ZT1jYWxsKCk7dG9yY2guY3VkYS5zeW5jaHJvbml6ZSgpO3JldHVybiB2YWx1ZSx0aW1lLnBlcmZfY291bnRlcigpLXN0YXJ0CiAgZGVmIHdhcm0oY2FsbCk6CiAgIGZvciBfIGluIHJhbmdlKFdBUk1VUCk6Y2FsbCgpCiAgIHRvcmNoLmN1ZGEuc3luY2hyb25pemUoKTt0aW1lcz1bXQogICBmb3IgXyBpbiByYW5nZShHUk9VUFMpOgogICAgYmVnaW4sZW5kPXRvcmNoLmN1ZGEuRXZlbnQoZW5hYmxlX3RpbWluZz1UcnVlKSx0b3JjaC5jdWRhLkV2ZW50KGVuYWJsZV90aW1pbmc9VHJ1ZSk7YmVnaW4ucmVjb3JkKCkKICAgIGZvciBfIGluIHJhbmdlKFJFUEVUSVRJT05TKTpjYWxsKCkKICAgIGVuZC5yZWNvcmQoKTtlbmQuc3luY2hyb25pemUoKTt0aW1lcy5hcHBlbmQoYmVnaW4uZWxhcHNlZF90aW1lKGVuZCkvUkVQRVRJVElPTlMpCiAgIGFzc2VydCBhbGwoeD4wIGZvciB4IGluIHRpbWVzKQogICByZXR1cm4geydjdWRhX2V2ZW50X21zX3Blcl9jYWxsJzp0aW1lcywnbWVkaWFuX21zJzpzdGF0aXN0aWNzLm1lZGlhbih0aW1lcyl9CiAgc2NyYXRjaD10b3JjaC5vbmVzKDE2LGRldmljZT0nY3VkYScpO3NjcmF0Y2guYWRkXygxKTt0b3JjaC5jdWRhLnN5bmNocm9uaXplKCk7ZGVsIHNjcmF0Y2gKICBmb3IgdCBpbiBUUzoKICAgeCxiLHc9Zml4dHVyZSg0KnQsZGV2aWNlPSdjdWRhJyk7ZHk9dG9yY2guZnVsbF9saWtlKHgsMTI4Lik7ZHo9dG9yY2guZnVsbF9saWtlKHgsLTY0Lik7cm5nPXRvcmNoLmN1ZGEuZ2V0X3JuZ19zdGF0ZSgpLmNsb25lKCkKICAgeSxmb3J3YXJkPXdhbGwobGFtYmRhOmZuKHgsYix3KSk7YmFja3dhcmQ9bGFtYmRhOnRvcmNoLmF1dG9ncmFkLmdyYWQoeSwoeCxiKSwoZHksZHopLHJldGFpbl9ncmFwaD1UcnVlKQogICBncmFkcyxiYWNrPXdhbGwoYmFja3dhcmQpO2V4cGVjdGVkPW5hdGl2ZSh4LGIsdyk7ZWc9dG9yY2guYXV0b2dyYWQuZ3JhZChleHBlY3RlZCwoeCxiKSwoZHksZHopKQogICBmZXJyb3JzPVtlbmdpbmVlcmluZ19jbG9zZShnLGUsZm9yd2FyZD1UcnVlKSBmb3IgZyxlIGluIHppcCh5LGV4cGVjdGVkKV07Z2Vycm9ycz1bZW5naW5lZXJpbmdfY2xvc2UoZyxlLHNjYWxlPTEyOC4pIGZvciBnLGUgaW4gemlwKGdyYWRzLGVnKV0KICAgYXNzZXJ0IHRvcmNoLmVxdWFsKHlbMV0sZXhwZWN0ZWRbMV0pO2Fzc2VydCB0b3JjaC5lcXVhbChybmcsdG9yY2guY3VkYS5nZXRfcm5nX3N0YXRlKCkpCiAgIHJvdz17J0InOjQsJ1QnOnQsJ0gnOjE1MzYsJ2R0eXBlJzonZmxvYXQzMicsJ2Zyb3plbl93ZWlnaHQnOlRydWUsJ2JvdGhfb3V0cHV0X2dyYWRpZW50cyc6VHJ1ZSwncmVzaWR1YWxfZXhhY3QnOlRydWUsJ2ZvcndhcmRfZXJyb3JzJzpmZXJyb3JzLCdncmFkaWVudF9lcnJvcnMnOmdlcnJvcnMsJ2NvbGRfZmlyc3RfY2FsbF93YWxsX3MnOnsnZm9yd2FyZCc6Zm9yd2FyZCwnYmFja3dhcmQnOmJhY2t9fQogICByb3dbJ3dhcm1fZm9yd2FyZCddPXdhcm0obGFtYmRhOmZuKHgsYix3KSk7cm93Wyd3YXJtX2JhY2t3YXJkJ109d2FybShiYWNrd2FyZCk7YXNzZXJ0IHRvcmNoLmVxdWFsKHJuZyx0b3JjaC5jdWRhLmdldF9ybmdfc3RhdGUoKSkKICAgcmVwb3J0Wydyb3dzJ10uYXBwZW5kKHJvdyk7d3JpdGUob3V0LHJlcG9ydCkKICAgZGVsIHksZ3JhZHMsZXhwZWN0ZWQsZWcsYmFja3dhcmQseCxiLHcsZHksZHoKICByZXBvcnQudXBkYXRlKHN0YXR1cz0nUEFTU0VEX01JQ1JPQkVOQ0hfT05MWScsZ3B1X2V4ZWN1dGVkPVRydWUsbmF0aXZlX3NvdXJjZV9ibG9iPUVYUEVDVEVEX1FXRU5fQkxPQikKIGV4Y2VwdCBCYXNlRXhjZXB0aW9uIGFzIGU6cmVwb3J0LnVwZGF0ZShlcnJvcj1yZXByKGUpLHRyYWNlYmFjaz10cmFjZWJhY2suZm9ybWF0X2V4YygpKQogd3JpdGUob3V0LHJlcG9ydCk7cHJpbnQoanNvbi5kdW1wcyh7J21ldGhvZCc6YS5tZXRob2QsJ3N0YXR1cyc6cmVwb3J0WydzdGF0dXMnXSwncm93cyc6bGVuKHJlcG9ydFsncm93cyddKX0pKTtyYWlzZSBTeXN0ZW1FeGl0KHJlcG9ydFsnc3RhdHVzJ10hPSdQQVNTRURfTUlDUk9CRU5DSF9PTkxZJykKaWYgX19uYW1lX189PSdfX21haW5fXyc6bWFpbigpCg==', 'driver.py': 'IiIiQm91bmRlZCBuYXRpdmUvc3RhbmRhbG9uZS9mdXNlZCByZXNpZHVhbC1ub3JtIE1JQ1JPQkVOQ0hfT05MWSBvcmNoZXN0cmF0aW9uLgoKTmV2ZXIgaW5zdGFsbHMgaW50byBTT1VQX1JFUE8gb3IgY2FsbHMgYSB0cmFpbmluZyBoYXJuZXNzLiBFeGlzdGluZyBzb3VyY2UvcnVudGltZSwKbG9jaywgcGlkZmQvc3VicmVhcGVyLCBxdWFyYW50aW5lIGFuZCBpZGxlLWNoZWNrIGhlbHBlcnMgYXJlIHByZXNlcnZlZCB1bmNoYW5nZWQuClRoZSBkZXBsb3llZCBzZXJ2aWNlJ3MgbGltaXRzIHJlbWFpbiBhdXRob3JpdGF0aXZlOyBlYWNoIEdQVSBjaGlsZCBnZXRzIDw9ODAgcy4KIiIiCmltcG9ydCBmY250bCwgaGFzaGxpYiwganNvbiwgb3MsIHN5cywgdGltZSwgdXVpZApmcm9tIHBhdGhsaWIgaW1wb3J0IFBhdGgKaW1wb3J0IHByb2Nlc3NfZ3VhcmQKaW1wb3J0IG93bmVkX3Byb2Nlc3NfdHJlZSBhcyB0cmVlCmZyb20gcmV2aWV3ZWRfc3VwcG9ydCBpbXBvcnQgdmVyaWZ5X29mZmljaWFsLCBpZGxlX2NoZWNrLCBFeHBvcnRlciwgZXhwb3J0X3NtYWxsLCByZWFkLCB3cml0ZSwgc2hhCgpMT0NLID0gUGF0aCgnL3RtcC9zb3VwLWdwdTAubG9jaycpCk1BUktFUiA9IFBhdGgoJy90bXAvc291cC1ncHUwLmxvY2suYmxvY2tlZCcpCgpkZWYgbWFpbigpOgogICAgaGVyZSA9IFBhdGgoX19maWxlX18pLnJlc29sdmUoKS5wYXJlbnQKICAgIGlmIG5vdCBvcy5lbnZpcm9uLmdldCgnU09VUF9SRVBPJykgb3Igbm90IG9zLmVudmlyb24uZ2V0KCdTT1VQX1JFU1VMVFMnKToKICAgICAgICByYWlzZSBTeXN0ZW1FeGl0KCdSZXF1aXJlcyBkZXBsb3llZCBTT1VQX1JFUE8vU09VUF9SRVNVTFRTLCBubyBtYW51YWwgZmFsbGJhY2snKQogICAgcmVwbywgcmVzdWx0cyA9IFBhdGgob3MuZW52aXJvblsnU09VUF9SRVBPJ10pLmFic29sdXRlKCksIFBhdGgob3MuZW52aXJvblsnU09VUF9SRVNVTFRTJ10pLmFic29sdXRlKCkKICAgIGlmIHJlcG8uaXNfc3ltbGluaygpIG9yIHJlc3VsdHMuaXNfc3ltbGluaygpIG9yIHJlcG8gPT0gcmVzdWx0czoKICAgICAgICByYWlzZSBTeXN0ZW1FeGl0KCdQcmVwYXJlZCBjaGVja291dC9yZXN1bHRzIG11c3QgYmUgc2VwYXJhdGUgcmVhbCBkaXJlY3RvcmllcycpCiAgICBhc3NlcnQgUGF0aCgnL2thZ2dsZS93b3JraW5nJykuaXNfZGlyKCkKICAgIHJlc3VsdHMubWtkaXIocGFyZW50cz1UcnVlLGV4aXN0X29rPVRydWUpCiAgICBpZGVudGl0eSA9ICdyZXNpZHVhbC1ub3JtLW1pY3JvYmVuY2gtb25seS0nICsgdXVpZC51dWlkNCgpLmhleFs6MTJdCiAgICBydW4sIGV4cG9ydCA9IFBhdGgoJy9rYWdnbGUvd29ya2luZycpL2lkZW50aXR5LCByZXN1bHRzL2lkZW50aXR5CiAgICBydW4ubWtkaXIoZXhpc3Rfb2s9RmFsc2UpOyBleHBvcnQubWtkaXIoZXhpc3Rfb2s9RmFsc2UpCiAgICBzdGF0dXMgPSBkaWN0KHN0YXR1cz0nUFJFUEFSSU5HX01JQ1JPQkVOQ0hfT05MWScsc2NvcGU9J01JQ1JPQkVOQ0hfT05MWScsCiAgICAgICAgICAgICAgICAgIHRyYWluaW5nX2V4ZWN1dGVkPUZhbHNlLGNvbXBsZXRlX2ZpeGVkX3Rhc2s9RmFsc2Usc2NvcmU9Tm9uZSxhZG1pc3Npb249J1VOS05PV04nLAogICAgICAgICAgICAgICAgICBtZXRob2RfdmFyaWFudHM9eyduYXRpdmUnOidleHBhbmRlZF9xd2VuX2FkZF9ub3JtJywnc3RhbmRhbG9uZSc6J2hpc3RvcmljYWxfb3JpZ2luYWxfdHJpYWwwMDFfbm9ybV9hZnRlcl9uYXRpdmVfYWRkJywnZnVzZWQnOidvcmlnaW5hbF90d29fb3V0cHV0X3Jlc2lkdWFsX25vcm0nfSwKICAgICAgICAgICAgICAgICAgcGVyX2dwdV9jaGlsZF93YWxsX2NhcF9zPTgwLGNhcF9ieXBhc3M9RmFsc2UsCiAgICAgICAgICAgICAgICAgIGNhbmRpZGF0ZV9zb3VyY2Vfc2hhMjU2PXJlYWQoaGVyZS8nc291cmNlX21hbmlmZXN0Lmpzb24nKSwKICAgICAgICAgICAgICAgICAgbWV0aG9kc19pbl9leGVjdXRpb25fb3JkZXI9WyduYXRpdmUnLCdzdGFuZGFsb25lJywnZnVzZWQnXSxwcm9jZXNzZXM9e30pCiAgICB3cml0ZShleHBvcnQvJ3N0YXR1cy5qc29uJyxzdGF0dXMpCiAgICBsb2NrZmQgPSBOb25lOyBzdGFydGVkID0gdGltZS5tb25vdG9uaWMoKQogICAgdHJ5OgogICAgICAgIGZvciBuYW1lLGRpZ2VzdCBpbiByZWFkKGhlcmUvJ3BheWxvYWRfbWFuaWZlc3QuanNvbicpLml0ZW1zKCk6CiAgICAgICAgICAgIHAgPSBoZXJlL25hbWUKICAgICAgICAgICAgYXNzZXJ0IG5vdCBwLmlzX3N5bWxpbmsoKSBhbmQgcC5pc19maWxlKCkgYW5kIHNoYShwKT09ZGlnZXN0LCAnUGF5bG9hZCBjaGFuZ2VkOiAnK25hbWUKICAgICAgICBpZiBNQVJLRVIuZXhpc3RzKCk6IHJhaXNlIFJ1bnRpbWVFcnJvcignR1BVIHF1YXJhbnRpbmUgcmVxdWlyZXMgb3BlcmF0b3IgcmVjb25jaWxpYXRpb24nKQogICAgICAgIGxvY2tmZCA9IG9zLm9wZW4oTE9DSyxvcy5PX0NSRUFUfG9zLk9fUkRXUnxvcy5PX05PRk9MTE9XLDBvNjAwKQogICAgICAgIGZjbnRsLmZsb2NrKGxvY2tmZCxmY250bC5MT0NLX0VYfGZjbnRsLkxPQ0tfTkIpCiAgICAgICAgaWYgTUFSS0VSLmV4aXN0cygpOiByYWlzZSBSdW50aW1lRXJyb3IoJ0dQVSBxdWFyYW50aW5lIGFwcGVhcmVkIHdoaWxlIGxvY2tpbmcnKQogICAgICAgIHRyZWUucHJlcGFyZSgpOyB0cmVlLmJlZ2luX3F1YXJhbnRpbmUoTUFSS0VSKQogICAgICAgIG9mZmljaWFsID0gcmVhZChoZXJlLydvZmZpY2lhbF9tYW5pZmVzdC5qc29uJykKICAgICAgICB2ZXJpZnlfb2ZmaWNpYWwocmVwbyxvZmZpY2lhbCkKICAgICAgICBlbnYgPSBkaWN0KG9zLmVudmlyb24sSEZfSFVCX09GRkxJTkU9JzEnLEhGX0RBVEFTRVRTX09GRkxJTkU9JzEnLFRSQU5TRk9STUVSU19PRkZMSU5FPScxJywKICAgICAgICAgICAgICAgICAgIFBZVEhPTkRPTlRXUklURUJZVEVDT0RFPScxJyxQWVRIT05VTkJVRkZFUkVEPScxJykKICAgICAgICBlbnYucG9wKCdTT1VQX1JPVEFSWV9DT0RFR0VOX0RJUicsTm9uZSkKICAgICAgICB3aXRoIEV4cG9ydGVyKHJ1bixleHBvcnQpOgogICAgICAgICAgICBwcm9iZSA9IHByb2Nlc3NfZ3VhcmQucnVuKFtzeXMuZXhlY3V0YWJsZSwnLUInLHN0cihoZXJlLydydW50aW1lX3Byb2JlLnB5Jyksc3RyKHJ1bi8ncnVudGltZV9wcm9iZS5qc29uJyldLAogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIGN3ZD1yZXBvLGVudj1lbnYsbG9nPXJ1bi8ncnVudGltZV9wcm9iZS5sb2cnLHRpbWVvdXQ9NDAsbG9ja19mZD1sb2NrZmQpCiAgICAgICAgICAgIHN0YXR1c1sncHJvY2Vzc2VzJ11bJ3J1bnRpbWVfcHJvYmUnXSA9IHByb2JlCiAgICAgICAgICAgIGlmIHByb2JlWydleGl0X2NvZGUnXSBvciBub3QgcmVhZChydW4vJ3J1bnRpbWVfcHJvYmUuanNvbicpLmdldCgncGFzc2VkJyk6CiAgICAgICAgICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoJ0V4YWN0IGRlcGxveWVkIHJ1bnRpbWUgZ2F0ZSBmYWlsZWQ7IG5vIGJlbmNobWFyaycpCiAgICAgICAgICAgIGRlZiBpZGxlKGxhYmVsKToKICAgICAgICAgICAgICAgIGZvbGRlcj1ydW4vKCdpZGxlXycrbGFiZWwpO2ZvbGRlci5ta2RpcihleGlzdF9vaz1GYWxzZSkKICAgICAgICAgICAgICAgIHJlc3VsdD1pZGxlX2NoZWNrKGVudixyZXBvLGZvbGRlcixsb2NrZmQpCiAgICAgICAgICAgICAgICBzdGF0dXNbJ3Byb2Nlc3NlcyddWydpZGxlXycrbGFiZWxdPXJlc3VsdAogICAgICAgICAgICAgICAgaWYgcmVzdWx0WydleGl0X2NvZGUnXTpyYWlzZSBSdW50aW1lRXJyb3IoJ0lkbGUgZ2F0ZSBmYWlsZWQ6ICcrbGFiZWwpCiAgICAgICAgICAgIGlkbGUoJ2JlZm9yZV9wcmVmbGlnaHQnKQogICAgICAgICAgICAjIEV4aXN0aW5nIG9yaWdpbmFsIG51bWVyaWNhbC9QVFggZ2F0ZSwgaXNvbGF0ZWQgZnJvbSBldmVyeSBtZWFzdXJlZCBjYWNoZS4KICAgICAgICAgICAgcHJlZW52ID0gZGljdChlbnYsVFJJVE9OX0NBQ0hFX0RJUj1zdHIocnVuLydwcmVmbGlnaHRfdHJpdG9uJyksCiAgICAgICAgICAgICAgICAgICAgICAgICAgVE9SQ0hJTkRVQ1RPUl9DQUNIRV9ESVI9c3RyKHJ1bi8ncHJlZmxpZ2h0X2luZHVjdG9yJykpCiAgICAgICAgICAgIHByZSA9IHByb2Nlc3NfZ3VhcmQucnVuKFtzeXMuZXhlY3V0YWJsZSwnLUInLCctbScsJ3NvbHV0aW9uLndlYl9wcmVmbGlnaHQnLCctLW91dCcsc3RyKHJ1bi8ncHJlZmxpZ2h0Lmpzb24nKV0sCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIGN3ZD1oZXJlLGVudj1wcmVlbnYsbG9nPXJ1bi8ncHJlZmxpZ2h0LmxvZycsdGltZW91dD04MCxsb2NrX2ZkPWxvY2tmZCkKICAgICAgICAgICAgc3RhdHVzWydwcm9jZXNzZXMnXVsncHJlZmxpZ2h0J10gPSBwcmUKICAgICAgICAgICAgaWYgcHJlWydleGl0X2NvZGUnXSBvciByZWFkKHJ1bi8ncHJlZmxpZ2h0Lmpzb24nKS5nZXQoJ3N0YXR1cycpIT0nUEFTU0VEX1BSRUZMSUdIVF9PTkxZJzoKICAgICAgICAgICAgICAgIHJhaXNlIFJ1bnRpbWVFcnJvcignT3JpZ2luYWwgcmVzaWR1YWwtbm9ybSBudW1lcmljYWwgZ2F0ZSBmYWlsZWQ7IG5vIGJlbmNobWFyaycpCiAgICAgICAgICAgIHN0YXR1c1snc3RhdHVzJ109J1JVTk5JTkdfTUlDUk9CRU5DSF9PTkxZJzt3cml0ZShleHBvcnQvJ3N0YXR1cy5qc29uJyxzdGF0dXMpCiAgICAgICAgICAgIGlkbGUoJ3ByZWZsaWdodCcpCiAgICAgICAgICAgIGZvciBtZXRob2QgaW4gc3RhdHVzWydtZXRob2RzX2luX2V4ZWN1dGlvbl9vcmRlciddOgogICAgICAgICAgICAgICAgY2FjaGU9cnVuLyd0cml0b25fY2FjaGUnL21ldGhvZDtjYWNoZS5ta2RpcihwYXJlbnRzPVRydWUsZXhpc3Rfb2s9RmFsc2UpCiAgICAgICAgICAgICAgICBjaGlsZGVudj1kaWN0KGVudixUUklUT05fQ0FDSEVfRElSPXN0cihjYWNoZSksVE9SQ0hJTkRVQ1RPUl9DQUNIRV9ESVI9c3RyKHJ1bi8naW5kdWN0b3JfY2FjaGUnL21ldGhvZCkpCiAgICAgICAgICAgICAgICBwcm9jZXNzPXByb2Nlc3NfZ3VhcmQucnVuKFtzeXMuZXhlY3V0YWJsZSwnLUInLHN0cihoZXJlLyd3b3JrZXIucHknKSwnLS1tZXRob2QnLG1ldGhvZCwnLS1vdXQnLHN0cihydW4vKG1ldGhvZCsnLmpzb24nKSldLAogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICBjd2Q9aGVyZSxlbnY9Y2hpbGRlbnYsbG9nPXJ1bi8obWV0aG9kKycubG9nJyksdGltZW91dD04MCxsb2NrX2ZkPWxvY2tmZCkKICAgICAgICAgICAgICAgIHN0YXR1c1sncHJvY2Vzc2VzJ11bbWV0aG9kXT1wcm9jZXNzCiAgICAgICAgICAgICAgICBpZiBwcm9jZXNzWydleGl0X2NvZGUnXSBvciByZWFkKHJ1bi8obWV0aG9kKycuanNvbicpKS5nZXQoJ3N0YXR1cycpIT0nUEFTU0VEX01JQ1JPQkVOQ0hfT05MWSc6CiAgICAgICAgICAgICAgICAgICAgcmFpc2UgUnVudGltZUVycm9yKCdJbmNvbXBsZXRlL2ZhaWxlZC9jZW5zb3JlZCBtaWNyb2JlbmNobWFyazogJyttZXRob2QpCiAgICAgICAgICAgICAgICBpZGxlKG1ldGhvZCkKICAgICAgICAgICAgICAgIHdyaXRlKGV4cG9ydC8nc3RhdHVzLmpzb24nLHN0YXR1cykKICAgICAgICAgICAgdmVyaWZ5X29mZmljaWFsKHJlcG8sb2ZmaWNpYWwpCiAgICAgICAgICAgIHJlcG9ydHM9e21ldGhvZDpyZWFkKHJ1bi8obWV0aG9kKycuanNvbicpKSBmb3IgbWV0aG9kIGluIHN0YXR1c1snbWV0aG9kc19pbl9leGVjdXRpb25fb3JkZXInXX0KICAgICAgICAgICAgYXNzZXJ0IGFsbChsZW4oclsncm93cyddKT09NSBhbmQgclsndHJhaW5pbmdfZXhlY3V0ZWQnXSBpcyBGYWxzZSBhbmQgclsnY29tcGxldGVfZml4ZWRfdGFzayddIGlzIEZhbHNlIGZvciByIGluIHJlcG9ydHMudmFsdWVzKCkpCiAgICAgICAgICAgIHdyaXRlKHJ1bi8nY29tcGFyaXNvbi5qc29uJyxkaWN0KHNjb3BlPSdNSUNST0JFTkNIX09OTFknLHJlcG9ydHM9cmVwb3J0cywKICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICBzY29yZT1Ob25lLGFkbWlzc2lvbj0nVU5LTk9XTicsY29tcGxldGVfZml4ZWRfdGFzaz1GYWxzZSwKICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICBub3RlPSdSYXcgZmlyc3QtY2FsbCB3YWxsIGFuZCB3YXJtIENVREEtZXZlbnQgc2FtcGxlcywgbm90IHB1cmUgY29tcGlsZSBjb3N0IG9yIHRhc2sgc3BlZWR1cDsgbm8gcmF0aW9zIG9yIG93bmVyc2hpcC1zaGFyZSBpbmZlcmVuY2UuJykpCiAgICAgICAgICAgIHN0YXR1c1snc3RhdHVzJ109J1BBU1NFRF9NSUNST0JFTkNIX09OTFknCiAgICBleGNlcHQgQmFzZUV4Y2VwdGlvbiBhcyBleGM6CiAgICAgICAgc3RhdHVzLnVwZGF0ZShzdGF0dXM9J0JMT0NLRURfRkFJTEVEX09SX0NFTlNPUkVEX01JQ1JPQkVOQ0hfT05MWScsZXJyb3I9cmVwcihleGMpKQogICAgICAgIGlmIGxvY2tmZCBpcyBub3QgTm9uZTp0cmVlLnF1YXJhbnRpbmUoJ1Jlc2lkdWFsLW5vcm0gbWljcm9iZW5jaG1hcmsgZmFpbGVkOiAnK3JlcHIoZXhjKSkKICAgIGZpbmFsbHk6CiAgICAgICAgc3RhdHVzWydvcmNoZXN0cmF0b3Jfd2FsbF9zJ109dGltZS5tb25vdG9uaWMoKS1zdGFydGVkCiAgICAgICAgZXhwb3J0X3NtYWxsKHJ1bixleHBvcnQpO3dyaXRlKGV4cG9ydC8nc3RhdHVzLmpzb24nLHN0YXR1cykKICAgICAgICBpZiBsb2NrZmQgaXMgbm90IE5vbmU6CiAgICAgICAgICAgIHRyZWUuZmluaXNoX3F1YXJhbnRpbmUoKTtvcy5jbG9zZShsb2NrZmQpCiAgICBwcmludChqc29uLmR1bXBzKHsnc2NvcGUnOidNSUNST0JFTkNIX09OTFknLCdzdGF0dXMnOnN0YXR1c1snc3RhdHVzJ10sJ2V4cG9ydCc6c3RyKGV4cG9ydCksCiAgICAgICAgICAgICAgICAgICAgICAnc2NvcmUnOk5vbmUsJ2FkbWlzc2lvbic6J1VOS05PV04nfSksZmx1c2g9VHJ1ZSkKICAgIHJhaXNlIFN5c3RlbUV4aXQoMCBpZiBzdGF0dXNbJ3N0YXR1cyddPT0nUEFTU0VEX01JQ1JPQkVOQ0hfT05MWScgZWxzZSAxKQoKaWYgX19uYW1lX189PSdfX21haW5fXyc6bWFpbigpCg==', 'source_manifest.json': 'ewogICJzb2x1dGlvbi9fX2luaXRfXy5weSI6ICI1ODg1YjRhNzg5MTAzZTE0YjhmZmIyMjgzZWQ1MDU4ZWVlMDVlYzQ2YjZiYmExYWZmNmI4YmUzZGU4YjBlYjExIiwKICAic29sdXRpb24vY29udHJhY3RzLnB5IjogImFjNTRjZDFkYTU5ZWNmMzhjOWI1ZjIyYjM4N2IxMTczZjMxYzRmMTk1OWQxZDViZjNjYWEzNWFkOWIyYzQ1NWYiLAogICJzb2x1dGlvbi9yZXNpZHVhbF9ub3JtLnB5IjogIjgwMjAzZjI1NTdiMGMzMTRjOGM1MjVmNWQ4Y2UwZTRjZjk4OWJmMGYyZWM4NWIwMzRjZDViNzFlNTM0NzA1MGEiLAogICJzb2x1dGlvbi93ZWJfcHJlZmxpZ2h0LnB5IjogImI1NWYxNTBjZGQzZGExMDkxMmUxMjc1NTc4MWEzNjdlYTc5YzZkYWQwYjM5MTEyZDAyZWQwNDMzMDY0ZDliYjkiLAogICJvbGRfc29sdXRpb24vX19pbml0X18ucHkiOiAiOTRlMTBhMTI3MWZlYzJhNDI5ZDY5ZTI4ZTRkOGIzYWRmYWM1OWJkZDMyM2MzYmYwZDBmYTNjZjYxNjUzNGIwZCIsCiAgIm9sZF9zb2x1dGlvbi9ybXNub3JtLnB5IjogIjk1ZjIyYThlMDkzMDlkMWQ2YmU1N2QzODlkNWRjOGZjOGZlYWEwNWJhNGE4NTEwMTkyNDQzYTk0MDA5YTlkZGUiCn0K', 'payload_manifest.json': 'ewogICJzb2x1dGlvbi9fX2luaXRfXy5weSI6ICI1ODg1YjRhNzg5MTAzZTE0YjhmZmIyMjgzZWQ1MDU4ZWVlMDVlYzQ2YjZiYmExYWZmNmI4YmUzZGU4YjBlYjExIiwKICAic29sdXRpb24vY29udHJhY3RzLnB5IjogImFjNTRjZDFkYTU5ZWNmMzhjOWI1ZjIyYjM4N2IxMTczZjMxYzRmMTk1OWQxZDViZjNjYWEzNWFkOWIyYzQ1NWYiLAogICJzb2x1dGlvbi9yZXNpZHVhbF9ub3JtLnB5IjogIjgwMjAzZjI1NTdiMGMzMTRjOGM1MjVmNWQ4Y2UwZTRjZjk4OWJmMGYyZWM4NWIwMzRjZDViNzFlNTM0NzA1MGEiLAogICJzb2x1dGlvbi93ZWJfcHJlZmxpZ2h0LnB5IjogImI1NWYxNTBjZGQzZGExMDkxMmUxMjc1NTc4MWEzNjdlYTc5YzZkYWQwYjM5MTEyZDAyZWQwNDMzMDY0ZDliYjkiLAogICJvbGRfc29sdXRpb24vX19pbml0X18ucHkiOiAiOTRlMTBhMTI3MWZlYzJhNDI5ZDY5ZTI4ZTRkOGIzYWRmYWM1OWJkZDMyM2MzYmYwZDBmYTNjZjYxNjUzNGIwZCIsCiAgIm9sZF9zb2x1dGlvbi9ybXNub3JtLnB5IjogIjk1ZjIyYThlMDkzMDlkMWQ2YmU1N2QzODlkNWRjOGZjOGZlYWEwNWJhNGE4NTEwMTkyNDQzYTk0MDA5YTlkZGUiLAogICJwcm9jZXNzX2d1YXJkLnB5IjogIjgyNDY1OTQzYTNhOWFmMzgzMWE1Mzk5YTNhNTVkMDlhYzI5MGJhNzM2NWM0ZTgwZDZiYzVlNzlmYWI2MmE2ZmYiLAogICJvd25lZF9wcm9jZXNzX3RyZWUucHkiOiAiNjkzZGRlNzA4ZGE1YzA2MDUzYWYyMzk0NjM5NTIxMTVlZjVlODg2NWY1MGJjYmM1MWEyMDM4MjM3Y2E1N2Q2OSIsCiAgInJ1bnRpbWVfcHJvYmUucHkiOiAiZWZlMWI3Y2YzZjNjZjFlMzdiYmMxNjlhODQ2ODg1OWM1MTI3ZWMzMjgxYTM3ZTM5MDY4NzAzYjJlYjNjMTQ4MCIsCiAgIm9mZmljaWFsX21hbmlmZXN0Lmpzb24iOiAiZDAxM2UxNjMzOTQwYzI3ZDcwOGUyNDc0YTgyODA5OTAwMDBlMzMwMThlODg5ZWQxNDY1NWM4NTYzMjlmZjFkOCIsCiAgInJldmlld2VkX3N1cHBvcnQucHkiOiAiN2M1ZDYwMmQwNzEzODNlZjA3ZmRkZDk5NjcwZWQxY2NlMzE0MWZlYWJkNWFmODM5NGYwNTNjNmI0ZTA3MTIyYyIsCiAgIndvcmtlci5weSI6ICIxZjQ5Y2I4OWJhMzhmOTNmYWYxYTYzYmEyY2MzZWNjOTVkYWYzOGU0YzcyYTJhOTdiMjNlMzU2OTJhMGJkNzI1IiwKICAiZHJpdmVyLnB5IjogImFlZjMwY2YxYTIyZTdlN2E5ZmM4MmY1OWQ2NWM4MzAyNjhhYTQ4ZjM4YjRjMGYyZTI4NDBiOTBhZTVmM2U3OTgiLAogICJzb3VyY2VfbWFuaWZlc3QuanNvbiI6ICI0ZjU2YWViYTY5MDFiMmVjYjI1NGQ5NjlkYmFiYzNjMzY3Mjk0Y2YwOTZmM2ExYjZiMTE5OTExN2YwMWQxYWI2Igp9Cg=='}
PAYLOAD_SHA256 = {'solution/__init__.py': '5885b4a789103e14b8ffb2283ed5058eee05ec46b6bba1aff6b8be3de8b0eb11', 'solution/contracts.py': 'ac54cd1da59ecf38c9b5f22b387b1173f31c4f1959d1d5bf3caa35ad9b2c455f', 'solution/residual_norm.py': '80203f2557b0c314c8c525f5d8ce0e4cf989bf0f2ec85b034cd5b71e5347050a', 'solution/web_preflight.py': 'b55f150cdd3da10912e12755781a367ea79c6dad0b39112d02ed0433064d9bb9', 'old_solution/__init__.py': '94e10a1271fec2a429d69e28e4d8b3adfac59bdd323c3bf0d0fa3cf616534b0d', 'old_solution/rmsnorm.py': '95f22a8e09309d1d6be57d389d5dc8fc8feaa05ba4a8510192443a94009a9dde', 'process_guard.py': '82465943a3a9af3831a5399a3a55d09ac290ba7365c4e80d6bc5e79fab62a6ff', 'owned_process_tree.py': '693dde708da5c06053af239463952115ef5e8865f50bcbc51a2038237ca57d69', 'runtime_probe.py': 'efe1b7cf3f3cf1e37bbc169a8468859c5127ec3281a37e39068703b2eb3c1480', 'official_manifest.json': 'd013e1633940c27d708e2474a8280990000e33018e889ed14655c856329ff1d8', 'reviewed_support.py': '7c5d602d071383ef07fddd99670ed1cce3141feabd5af8394f053c6b4e07122c', 'worker.py': '1f49cb89ba38f93faf1a63ba2cc3ecc95daf38e4c72a2a97b23e35692a0bd725', 'driver.py': 'aef30cf1a22e7e7a9fc82f59d65c830268aa48f38b4c0f2e2840b90ae5f3e798', 'source_manifest.json': '4f56aeba6901b2ecb254d969dbabc3c367294cf096f3a1b6b1199117f01d1ab6', 'payload_manifest.json': 'cb0cd914797d771d6a0c2c53f7d9a1e345542a08e844e2b77ab638f64f5a4d78'}
STAGED=Path(tempfile.mkdtemp(prefix='residual-norm-microbench-',dir='/kaggle/working'))
for name,encoded in PAYLOAD.items():
    parts=Path(name).parts
    assert not Path(name).is_absolute() and '..' not in parts
    raw=base64.b64decode(encoded,validate=True)
    assert hashlib.sha256(raw).hexdigest()==PAYLOAD_SHA256[name]
    target=STAGED/name;target.parent.mkdir(parents=True,exist_ok=True);target.write_bytes(raw)
print('Prepared MICROBENCH_ONLY payload:',STAGED)


In [ ]:
import subprocess,sys,signal
process=subprocess.Popen([sys.executable,'-B',str(STAGED/'driver.py')],start_new_session=True)
try:
    return_code=process.wait()
except BaseException:
    if process.poll() is None:process.send_signal(signal.SIGTERM)
    while process.poll() is None:
        try:process.wait()
        except KeyboardInterrupt:pass
    raise
if return_code:raise RuntimeError('MICROBENCH_ONLY failed or censored; inspect exported status. No automatic retry.')
